# データ保存

In [ ]:
import os
os.environ["SPEDAS_DATA_DIR"] = "/mnt/j/observation_data/"

# PWE-EFDのスペクトルを見てみる

In [ ]:
import pyspedas as psp
import pytplot as pt
import ergpyspedas.erg as ergpy

pt.del_data('*')

time_range = ['20210401/00:00:00', '20210501/00:00:00']

ergpy.pwe_efd(trange=time_range, level='l2', datatype='spec', get_support_data=True)

In [ ]:
efd_spectra         = psp.get_data('erg_pwe_efd_l2_spec_spectra', xarray=True).sortby('time')
efd_spectra_flag    = psp.get_data('erg_pwe_efd_l2_spec_quality_flag', xarray=True).sortby('time')

bad_times = efd_spectra_flag.time.where(efd_spectra_flag != 0, drop=True)

print(efd_spectra)
print('')
print(efd_spectra_flag)
print('')
print(bad_times)

In [ ]:
import numpy as np

qf_on_spec = efd_spectra_flag.reindex(
    time=efd_spectra.time,
    method="nearest",
    tolerance=np.timedelta64(500, "ms")
)

efd_spectra_qf = efd_spectra.where(qf_on_spec == 0, np.nan)

# LEP-i, LEP-eのデータ存在時間の確認

In [ ]:
ergpy.lepe(trange=time_range, datatype='omniflux', level='l2')
ergpy.lepi(trange=time_range, datatype='omniflux', level='l2')

In [ ]:
LEPe_FEDO_omniflux = psp.get_data('erg_lepe_l2_omniflux_FEDO', xarray=True).sortby('time')
LEPi_FPDO_omniflux = psp.get_data('erg_lepi_l2_omniflux_FPDO', xarray=True).sortby('time')

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr


def merge_omniflux_gap_ranges(
    gap_ranges_df,
    merge_gap_tolerance=pd.Timedelta(seconds=0),
):
    """
    gap_start_time/gap_end_time が重なる、接する、または
    merge_gap_tolerance 以下の短い good island を挟む gap 行を merge する。

    reason は統合元が分かるように '+' で連結する。
    """

    columns = [
        "gap_start_time",
        "gap_end_time",
        "gap_duration_sec",
        "gap_duration_min",
        "reason",
        "bad_start_time",
        "bad_end_time",
        "bad_duration_sec",
        "bad_duration_min",
        "n_bad_time_samples",
        "n_invalid_values",
        "invalid_fraction_max",
        "invalid_fraction_mean",
    ]

    if gap_ranges_df is None or len(gap_ranges_df) == 0:
        return pd.DataFrame(columns=columns)

    df = gap_ranges_df.copy()
    df["gap_start_time"] = pd.to_datetime(df["gap_start_time"])
    df["gap_end_time"] = pd.to_datetime(df["gap_end_time"])
    df = df.dropna(subset=["gap_start_time", "gap_end_time"])
    df = df[df["gap_end_time"] >= df["gap_start_time"]]
    df = df.sort_values(["gap_start_time", "gap_end_time"]).reset_index(drop=True)

    if len(df) == 0:
        return pd.DataFrame(columns=columns)

    merge_gap_tolerance = pd.Timedelta(merge_gap_tolerance)

    groups = []
    current = [0]
    current_end = df.loc[0, "gap_end_time"]

    for idx in range(1, len(df)):
        start = df.loc[idx, "gap_start_time"]
        end = df.loc[idx, "gap_end_time"]

        if start <= current_end + merge_gap_tolerance:
            current.append(idx)
            if end > current_end:
                current_end = end
        else:
            groups.append(current)
            current = [idx]
            current_end = end

    groups.append(current)

    merged_rows = []

    for group in groups:
        g = df.loc[group]
        gap_start = g["gap_start_time"].min()
        gap_end = g["gap_end_time"].max()
        gap_dt = gap_end - gap_start

        reasons = []
        for reason in g["reason"].dropna().astype(str):
            for part in reason.split("+"):
                if part not in reasons:
                    reasons.append(part)

        bad_starts = pd.to_datetime(g["bad_start_time"], errors="coerce").dropna()
        bad_ends = pd.to_datetime(g["bad_end_time"], errors="coerce").dropna()

        n_bad = pd.to_numeric(g["n_bad_time_samples"], errors="coerce").fillna(0)
        n_invalid = pd.to_numeric(g["n_invalid_values"], errors="coerce").fillna(0)
        invalid_max = pd.to_numeric(g["invalid_fraction_max"], errors="coerce")
        invalid_mean = pd.to_numeric(g["invalid_fraction_mean"], errors="coerce")

        valid_mean = invalid_mean.notna() & (n_bad > 0)
        if valid_mean.any():
            invalid_fraction_mean = float(
                np.average(invalid_mean[valid_mean], weights=n_bad[valid_mean])
            )
        else:
            invalid_fraction_mean = np.nan

        bad_duration = pd.to_numeric(g["bad_duration_sec"], errors="coerce").sum(min_count=1)

        merged_rows.append({
            "gap_start_time": gap_start,
            "gap_end_time": gap_end,
            "gap_duration_sec": gap_dt.total_seconds(),
            "gap_duration_min": gap_dt.total_seconds() / 60,
            "reason": "+".join(reasons),
            "bad_start_time": bad_starts.min() if len(bad_starts) > 0 else pd.NaT,
            "bad_end_time": bad_ends.max() if len(bad_ends) > 0 else pd.NaT,
            "bad_duration_sec": float(bad_duration) if pd.notna(bad_duration) else np.nan,
            "bad_duration_min": float(bad_duration) / 60 if pd.notna(bad_duration) else np.nan,
            "n_bad_time_samples": int(n_bad.sum()),
            "n_invalid_values": int(n_invalid.sum()),
            "invalid_fraction_max": float(invalid_max.max()) if invalid_max.notna().any() else np.nan,
            "invalid_fraction_mean": invalid_fraction_mean,
        })

    return pd.DataFrame(merged_rows, columns=columns).reset_index(drop=True)


def find_time_gaps_with_bad_omniflux(
    flux_da,
    threshold=pd.Timedelta(minutes=1),
    min_bad_duration=pd.Timedelta(seconds=30),
    invalid_fraction_threshold=0.3,
    time_dim="time",
    include_nonfinite=True,
    include_negative=True,
    merge_gap_ranges=True,
    merge_gap_tolerance=pd.Timedelta(seconds=0),
):
    """
    time gap に加えて、
    omniflux の使用不可データ割合が invalid_fraction_threshold 以上の状態が
    min_bad_duration 以上継続した場合に bad time gap として抽出する。

    Parameters
    ----------
    flux_da : xr.DataArray
        例: erg_lepe_l2_omniflux_FEDO
        dims は通常 (time, energy) を想定。
    threshold : pd.Timedelta
        隣接時刻差がこの値以上なら通常の time gap とみなす。
    min_bad_duration : pd.Timedelta
        bad 状態がこの時間以上継続した場合のみ gap として採用する。
    invalid_fraction_threshold : float
        各時刻で使用不可データがこの割合以上なら bad sample とする。
        例: 0.3 なら 30%以上。
    time_dim : str
        時間次元名。
    include_nonfinite : bool
        True の場合、NaN/inf を使用不可とみなす。
    include_negative : bool
        True の場合、負値を使用不可とみなす。
    merge_gap_ranges : bool
        True の場合、検出後に重なる/接する gap 行を merge する。
    merge_gap_tolerance : pd.Timedelta
        gap 同士の間にこの時間以下の短い good island がある場合も merge する。
        0秒なら重なりまたは接している区間だけを merge する。

    Returns
    -------
    gap_ranges_df : pd.DataFrame
    """

    da = flux_da.sortby(time_dim)
    times = pd.DatetimeIndex(pd.to_datetime(da[time_dim].values))

    columns = [
        "gap_start_time",
        "gap_end_time",
        "gap_duration_sec",
        "gap_duration_min",
        "reason",
        "bad_start_time",
        "bad_end_time",
        "bad_duration_sec",
        "bad_duration_min",
        "n_bad_time_samples",
        "n_invalid_values",
        "invalid_fraction_max",
        "invalid_fraction_mean",
    ]

    if len(times) == 0:
        return pd.DataFrame(columns=columns)

    rows = []

    # ============================================================
    # 1. 通常の time gap
    # ============================================================
    if len(times) >= 2:
        dt = times[1:] - times[:-1]
        gap_mask = dt >= threshold

        for i in np.where(gap_mask)[0]:
            rows.append({
                "gap_start_time": times[i],
                "gap_end_time": times[i + 1],
                "gap_duration_sec": dt[i].total_seconds(),
                "gap_duration_min": dt[i].total_seconds() / 60,
                "reason": "time_gap",
                "bad_start_time": pd.NaT,
                "bad_end_time": pd.NaT,
                "bad_duration_sec": np.nan,
                "bad_duration_min": np.nan,
                "n_bad_time_samples": 0,
                "n_invalid_values": 0,
                "invalid_fraction_max": np.nan,
                "invalid_fraction_mean": np.nan,
            })

    # ============================================================
    # 2. 使用不可データ割合を評価
    # ============================================================
    reduce_dims = [d for d in da.dims if d != time_dim]

    invalid_da = xr.zeros_like(da, dtype=bool)

    if include_nonfinite:
        invalid_da = invalid_da | (~np.isfinite(da))

    if include_negative:
        invalid_da = invalid_da | (da < 0)

    if len(reduce_dims) > 0:
        n_invalid_each_time = invalid_da.sum(dim=reduce_dims)
        n_total_each_time = xr.ones_like(invalid_da, dtype=int).sum(dim=reduce_dims)

        invalid_fraction = n_invalid_each_time / n_total_each_time
        bad_time = invalid_fraction >= invalid_fraction_threshold

        n_invalid_each_time = n_invalid_each_time.values.astype(int)
        invalid_fraction_each_time = invalid_fraction.values.astype(float)
        bad_time = bad_time.values.astype(bool)

    else:
        n_invalid_each_time = invalid_da.values.astype(int)
        invalid_fraction_each_time = invalid_da.values.astype(float)
        bad_time = invalid_fraction_each_time >= invalid_fraction_threshold

    bad_indices = np.where(bad_time)[0]

    if len(bad_indices) > 0:
        # 連続する bad sample を block 化
        split_points = np.where(np.diff(bad_indices) > 1)[0]

        block_starts = np.r_[bad_indices[0], bad_indices[split_points + 1]]
        block_ends   = np.r_[bad_indices[split_points], bad_indices[-1]]

        for i0, i1 in zip(block_starts, block_ends):

            bad_start = times[i0]

            # bad interval の終端は最後の bad sample の次時刻とする
            if i1 < len(times) - 1:
                bad_interval_end = times[i1 + 1]
            else:
                if len(times) >= 2:
                    cadence = pd.Series(times[1:] - times[:-1]).median()
                else:
                    cadence = pd.Timedelta(0)

                bad_interval_end = times[i1] + cadence

            bad_duration = bad_interval_end - bad_start

            # 継続時間が足りなければ採用しない
            if bad_duration < min_bad_duration:
                continue

            # gap としては直前の good sample から直後の good sample まで
            if i0 > 0:
                gap_start = times[i0 - 1]
            else:
                gap_start = times[i0]

            if i1 < len(times) - 1:
                gap_end = times[i1 + 1]
            else:
                gap_end = bad_interval_end

            gap_dt = gap_end - gap_start

            invalid_frac_block = invalid_fraction_each_time[i0:i1 + 1]

            rows.append({
                "gap_start_time": gap_start,
                "gap_end_time": gap_end,
                "gap_duration_sec": gap_dt.total_seconds(),
                "gap_duration_min": gap_dt.total_seconds() / 60,
                "reason": "bad_omniflux_fraction",
                "bad_start_time": bad_start,
                "bad_end_time": times[i1],
                "bad_duration_sec": bad_duration.total_seconds(),
                "bad_duration_min": bad_duration.total_seconds() / 60,
                "n_bad_time_samples": int(i1 - i0 + 1),
                "n_invalid_values": int(n_invalid_each_time[i0:i1 + 1].sum()),
                "invalid_fraction_max": float(np.nanmax(invalid_frac_block)),
                "invalid_fraction_mean": float(np.nanmean(invalid_frac_block)),
            })

    gap_ranges_df = pd.DataFrame(rows, columns=columns)

    if len(gap_ranges_df) == 0:
        return gap_ranges_df

    gap_ranges_df = gap_ranges_df.sort_values(
        ["gap_start_time", "gap_end_time"]
    ).reset_index(drop=True)

    if merge_gap_ranges:
        gap_ranges_df = merge_omniflux_gap_ranges(
            gap_ranges_df,
            merge_gap_tolerance=merge_gap_tolerance,
        )

    return gap_ranges_df


In [ ]:
LEPe_omniflux_gap_ranges_df = find_time_gaps_with_bad_omniflux(
    LEPe_FEDO_omniflux,
    threshold=pd.Timedelta(minutes=1),
    min_bad_duration=pd.Timedelta(minutes=1),
    invalid_fraction_threshold=0.3,
)

print(f"1分以上の time gap 数: {len(LEPe_omniflux_gap_ranges_df)}")
display(LEPe_omniflux_gap_ranges_df)

In [ ]:
LEPi_omniflux_gap_ranges_df = find_time_gaps_with_bad_omniflux(
    LEPi_FPDO_omniflux,
    threshold=pd.Timedelta(minutes=1),
    min_bad_duration=pd.Timedelta(minutes=1),
    invalid_fraction_threshold=0.3,
)

print(f"1分以上の time gap 数: {len(LEPi_omniflux_gap_ranges_df)}")
display(LEPi_omniflux_gap_ranges_df)

# `erg_att_izras`と`erg_att_izdec`を取得

In [ ]:
import re
import numpy as np
import pandas as pd
import pyspedas as psp
from pyspedas import time_float, store_data
from ergpyspedas.erg.satellite.erg.load import load as erg_load


def load_att_robust(
    trange,
    level="l2",
    no_update=False,
    uname=None,
    passwd=None,
):
    """
    ERG/Arase attitude txt を robust に読み込んで tplot 変数へ保存する。

    ergpyspedas 標準の att loader は空白 split に依存しているため、
    例: "26.2539-151515.3574" のように隣接数値間の空白が欠けた行で落ちる。
    ここでは timestamp 以降の数値を正規表現で抽出し、必要列だけを使う。
    元txtファイルは変更しない。
    """

    file_res = 24 * 3600.0
    pathformat = f"satellite/erg/att/txt/erg_att_{level}_%Y%m%d_v??.txt"

    out_files = erg_load(
        pathformat=pathformat,
        trange=trange,
        file_res=file_res,
        downloadonly=True,
        no_update=no_update,
        uname=uname,
        passwd=passwd,
    )

    if out_files is None or len(out_files) == 0:
        raise FileNotFoundError(f"No ERG attitude files found for trange={trange}")

    # timestamp 以外の数値を抽出する。隣接した負値も別トークンとして拾う。
    float_pattern = re.compile(
        r"[+-]?(?:\d+\.\d*|\.\d+|\d+)(?:[Ee][+-]?\d+)?"
    )

    rows = []
    skipped = []
    fixed_joined_numeric = 0

    for file in out_files:
        with open(file, "r", encoding="utf-8", errors="replace") as f:
            lines = f.readlines()

        # ergpyspedas 標準 loader と同じく先頭10行はヘッダ扱い。
        for line_no, line in enumerate(lines[10:], start=11):
            stripped = line.strip()
            if not stripped:
                continue

            parts = stripped.split(maxsplit=1)
            if len(parts) < 2:
                skipped.append((file, line_no, "no payload", stripped))
                continue

            tstr, payload = parts
            nums = [float(x) for x in float_pattern.findall(payload)]

            # 標準 loader が使う列に対応するには最低11個の数値が必要。
            # nums: 0 Omega, 1 I_alpha, 2 I_delta, ..., 6 Phase,
            #       7 GX_alpha, 8 GX_delta, 9 GZ_alpha, 10 GZ_delta
            if len(nums) < 11:
                skipped.append((file, line_no, f"too few numeric values: {len(nums)}", stripped))
                continue

            # 空白欠落の典型例を集計だけしておく。
            if re.search(r"\d\.\d+-\d", payload):
                fixed_joined_numeric += 1

            rows.append({
                "time": tstr,
                "Omega": nums[0],
                "I_Alpha": nums[1],
                "I_Delta": nums[2],
                "Phase": nums[6],
                "GX_Alpha": nums[7],
                "GX_Delta": nums[8],
                "GZ_Alpha": nums[9],
                "GZ_Delta": nums[10],
            })

    if len(rows) == 0:
        raise ValueError("No valid attitude rows parsed.")

    att_df = pd.DataFrame(rows)
    att_df = att_df.drop_duplicates(subset="time").sort_values("time").reset_index(drop=True)

    t = time_float(att_df["time"].tolist())

    store_data("erg_att_sprate", data={"x": t, "y": att_df["Omega"].to_numpy(float)})
    store_data("erg_att_spphase", data={"x": t, "y": att_df["Phase"].to_numpy(float)})
    store_data("erg_att_izras", data={"x": t, "y": att_df["I_Alpha"].to_numpy(float)})
    store_data("erg_att_izdec", data={"x": t, "y": att_df["I_Delta"].to_numpy(float)})
    store_data("erg_att_gxras", data={"x": t, "y": att_df["GX_Alpha"].to_numpy(float)})
    store_data("erg_att_gxdec", data={"x": t, "y": att_df["GX_Delta"].to_numpy(float)})
    store_data("erg_att_gzras", data={"x": t, "y": att_df["GZ_Alpha"].to_numpy(float)})
    store_data("erg_att_gzdec", data={"x": t, "y": att_df["GZ_Delta"].to_numpy(float)})

    print(f"loaded attitude files: {len(out_files)}")
    print(f"parsed attitude rows: {len(att_df)}")
    print(f"fixed joined numeric rows: {fixed_joined_numeric}")
    print(f"skipped rows: {len(skipped)}")
    if len(skipped) > 0:
        print("first skipped row:", skipped[0])

    return att_df


att_df = load_att_robust(time_range, no_update=False)

# dsi2j2000.py / erg_interpolate_att.py と同じ処理
psp.degap("erg_att_izras", dt=8.0, margin=0.5)
psp.degap("erg_att_izdec", dt=8.0, margin=0.5)

att_izras = psp.get_data("erg_att_izras", xarray=True)
att_izdec = psp.get_data("erg_att_izdec", xarray=True)

print(att_izras)
print(att_izdec)


In [ ]:
def merge_att_nan_intervals(
    att_nan_intervals_df,
    merge_gap_tolerance=pd.Timedelta(seconds=0),
):
    """
    gap_start_time/gap_end_time が重なる、接する、または
    merge_gap_tolerance 以下の短い good island を挟む attitude NaN 区間を merge する。
    """

    columns = [
        "gap_start_time",
        "gap_end_time",
        "gap_duration_sec",
        "gap_duration_min",
        "bad_start_time",
        "bad_end_time",
        "bad_duration_sec",
        "bad_duration_min",
        "n_bad_time_samples",
        "n_nan_izras",
        "n_nan_izdec",
        "reason",
    ]

    if att_nan_intervals_df is None or len(att_nan_intervals_df) == 0:
        return pd.DataFrame(columns=columns)

    df = att_nan_intervals_df.copy()
    df["gap_start_time"] = pd.to_datetime(df["gap_start_time"])
    df["gap_end_time"] = pd.to_datetime(df["gap_end_time"])
    df = df.dropna(subset=["gap_start_time", "gap_end_time"])
    df = df[df["gap_end_time"] >= df["gap_start_time"]]
    df = df.sort_values(["gap_start_time", "gap_end_time"]).reset_index(drop=True)

    if len(df) == 0:
        return pd.DataFrame(columns=columns)

    merge_gap_tolerance = pd.Timedelta(merge_gap_tolerance)

    groups = []
    current = [0]
    current_end = df.loc[0, "gap_end_time"]

    for idx in range(1, len(df)):
        start = df.loc[idx, "gap_start_time"]
        end = df.loc[idx, "gap_end_time"]

        if start <= current_end + merge_gap_tolerance:
            current.append(idx)
            if end > current_end:
                current_end = end
        else:
            groups.append(current)
            current = [idx]
            current_end = end

    groups.append(current)

    merged_rows = []

    for group in groups:
        g = df.loc[group]
        gap_start = g["gap_start_time"].min()
        gap_end = g["gap_end_time"].max()
        gap_duration = gap_end - gap_start

        reasons = []
        for reason in g["reason"].dropna().astype(str):
            for part in reason.split("+"):
                if part not in reasons:
                    reasons.append(part)

        bad_starts = pd.to_datetime(g["bad_start_time"], errors="coerce").dropna()
        bad_ends = pd.to_datetime(g["bad_end_time"], errors="coerce").dropna()
        bad_duration = pd.to_numeric(g["bad_duration_sec"], errors="coerce").sum(min_count=1)

        n_bad = pd.to_numeric(g["n_bad_time_samples"], errors="coerce").fillna(0)
        n_nan_izras = pd.to_numeric(g["n_nan_izras"], errors="coerce").fillna(0)
        n_nan_izdec = pd.to_numeric(g["n_nan_izdec"], errors="coerce").fillna(0)

        merged_rows.append({
            "gap_start_time": gap_start,
            "gap_end_time": gap_end,
            "gap_duration_sec": gap_duration.total_seconds(),
            "gap_duration_min": gap_duration.total_seconds() / 60,
            "bad_start_time": bad_starts.min() if len(bad_starts) > 0 else pd.NaT,
            "bad_end_time": bad_ends.max() if len(bad_ends) > 0 else pd.NaT,
            "bad_duration_sec": float(bad_duration) if pd.notna(bad_duration) else np.nan,
            "bad_duration_min": float(bad_duration) / 60 if pd.notna(bad_duration) else np.nan,
            "n_bad_time_samples": int(n_bad.sum()),
            "n_nan_izras": int(n_nan_izras.sum()),
            "n_nan_izdec": int(n_nan_izdec.sum()),
            "reason": "+".join(reasons),
        })

    return pd.DataFrame(merged_rows, columns=columns).reset_index(drop=True)


def find_nan_intervals_att(
    att_izras,
    att_izdec,
    time_dim="time",
    merge_gap_ranges=True,
    merge_gap_tolerance=pd.Timedelta(seconds=0),
):
    """
    erg_att_izras / erg_att_izdec のどちらかが NaN になる連続区間を抽出する。
    gap_start_time / gap_end_time は、NaN block を挟む直前・直後の正常 grid。

    merge_gap_ranges=True の場合、重なる/接する gap 行を merge する。
    merge_gap_tolerance を指定すると、その時間以下の短い good island もまとめる。
    """

    izras = att_izras.sortby(time_dim)
    izdec = att_izdec.sortby(time_dim)

    times = pd.DatetimeIndex(pd.to_datetime(izras[time_dim].values))

    ras = np.asarray(izras.values)
    dec = np.asarray(izdec.values)

    bad = np.isnan(ras) | np.isnan(dec)
    bad_indices = np.where(bad)[0]

    columns = [
        "gap_start_time",
        "gap_end_time",
        "gap_duration_sec",
        "gap_duration_min",
        "bad_start_time",
        "bad_end_time",
        "bad_duration_sec",
        "bad_duration_min",
        "n_bad_time_samples",
        "n_nan_izras",
        "n_nan_izdec",
        "reason",
    ]

    if len(bad_indices) == 0:
        return pd.DataFrame(columns=columns)

    rows = []

    split_points = np.where(np.diff(bad_indices) > 1)[0]
    block_starts = np.r_[bad_indices[0], bad_indices[split_points + 1]]
    block_ends = np.r_[bad_indices[split_points], bad_indices[-1]]

    cadence = pd.Series(times[1:] - times[:-1]).median() if len(times) >= 2 else pd.Timedelta(0)

    for i0, i1 in zip(block_starts, block_ends):
        bad_start = times[i0]

        if i1 < len(times) - 1:
            bad_interval_end = times[i1 + 1]
        else:
            bad_interval_end = times[i1] + cadence

        bad_duration = bad_interval_end - bad_start

        # NaN block を挟む正常 grid の間を gap とする
        gap_start = times[i0 - 1] if i0 > 0 else times[i0]
        gap_end = times[i1 + 1] if i1 < len(times) - 1 else bad_interval_end
        gap_duration = gap_end - gap_start

        rows.append({
            "gap_start_time": gap_start,
            "gap_end_time": gap_end,
            "gap_duration_sec": gap_duration.total_seconds(),
            "gap_duration_min": gap_duration.total_seconds() / 60,
            "bad_start_time": bad_start,
            "bad_end_time": times[i1],
            "bad_duration_sec": bad_duration.total_seconds(),
            "bad_duration_min": bad_duration.total_seconds() / 60,
            "n_bad_time_samples": int(i1 - i0 + 1),
            "n_nan_izras": int(np.isnan(ras[i0:i1 + 1]).sum()),
            "n_nan_izdec": int(np.isnan(dec[i0:i1 + 1]).sum()),
            "reason": "att_izras_or_izdec_nan",
        })

    att_nan_intervals_df = pd.DataFrame(rows, columns=columns)

    if merge_gap_ranges:
        att_nan_intervals_df = merge_att_nan_intervals(
            att_nan_intervals_df,
            merge_gap_tolerance=merge_gap_tolerance,
        )

    return att_nan_intervals_df


In [ ]:
att_nan_intervals_df = find_nan_intervals_att(att_izras, att_izdec)

print(f"time gap 数: {len(att_nan_intervals_df)}")
display(att_nan_intervals_df)

# 第一段階検証 Especについて

In [ ]:
import numpy as np
import pandas as pd

# ============================================================
# gap range を time mask に変換する関数
# ============================================================

def make_not_in_gap_mask(times, gap_ranges_df_list):
    """
    times が、複数の gap_ranges_df のどの時間範囲にも含まれない mask を返す。

    gap_ranges_df は以下のどちらかの列を持つ想定:
      - missing_start_time, missing_end_time
      - gap_start_time, gap_end_time
    """

    times_pd = pd.DatetimeIndex(pd.to_datetime(times))

    in_gap = np.zeros(len(times_pd), dtype=bool)

    for gap_df in gap_ranges_df_list:
        if gap_df is None or len(gap_df) == 0:
            continue

        if {"missing_start_time", "missing_end_time"}.issubset(gap_df.columns):
            start_col = "missing_start_time"
            end_col   = "missing_end_time"
        elif {"gap_start_time", "gap_end_time"}.issubset(gap_df.columns):
            start_col = "gap_start_time"
            end_col   = "gap_end_time"
        else:
            raise ValueError(
                "gap_ranges_df must have either "
                "['missing_start_time', 'missing_end_time'] or "
                "['gap_start_time', 'gap_end_time'] columns."
            )

        for _, row in gap_df.iterrows():
            start = pd.to_datetime(row[start_col])
            end   = pd.to_datetime(row[end_col])

            if pd.isna(start) or pd.isna(end):
                continue

            in_gap |= (times_pd >= start) & (times_pd <= end)

    not_in_gap = ~in_gap

    return not_in_gap

In [ ]:
# spec_bins = 3..32 の範囲で、各 time ごとに 60% 以上のデータが 1e-4 を超える time を抽出
subset = efd_spectra_qf.sel(spec_bins=slice(3, 32))

valid = subset.notnull()
above = valid & (subset > 1e-4)

frac_above = above.sum(dim="v_dim") / valid.sum(dim="v_dim")
mask = frac_above >= 0.6

# ============================================================
# 3点以上連続している区間を丸ごと残す
# ============================================================

mask_np = mask.values.astype(bool)

kernel = np.ones(3, dtype=int)
count3 = np.convolve(mask_np.astype(int), kernel, mode="valid")
window3 = count3 == 3

mask_consecutive_3_np = np.zeros_like(mask_np, dtype=bool)

for i, ok in enumerate(window3):
    if ok:
        mask_consecutive_3_np[i:i+3] = True

# ============================================================
# LEPe / LEPi gap 範囲に含まれない条件を追加
# ============================================================

not_in_lep_gap_mask = make_not_in_gap_mask(
    frac_above.time.values,
    [
        LEPe_omniflux_gap_ranges_df,
        LEPi_omniflux_gap_ranges_df,
        att_nan_intervals_df,
    ],
)

# 最終 mask
final_mask_np = mask_consecutive_3_np & not_in_lep_gap_mask

selected_times = frac_above.time.values[final_mask_np]
selected_fracs = frac_above.values[final_mask_np]

print(f"3点以上連続条件を満たす time の数: {mask_consecutive_3_np.sum()}")
print(f"LEPe/LEPi gap 除外後の time の数: {len(selected_times)}")

In [ ]:
import numpy as np
import pandas as pd


def gap_df_to_intervals_fast(gap_df):
    """
    LEPe_gap_ranges_df / LEPi_gap_ranges_df を interval DataFrame に変換する。
    """

    if gap_df is None or len(gap_df) == 0:
        return pd.DataFrame(columns=["start_time", "end_time"])

    if {"missing_start_time", "missing_end_time"}.issubset(gap_df.columns):
        start_col = "missing_start_time"
        end_col = "missing_end_time"
    elif {"gap_start_time", "gap_end_time"}.issubset(gap_df.columns):
        start_col = "gap_start_time"
        end_col = "gap_end_time"
    else:
        raise ValueError(
            "gap_df must have either "
            "['missing_start_time', 'missing_end_time'] or "
            "['gap_start_time', 'gap_end_time'] columns."
        )

    out = pd.DataFrame({
        "start_time": pd.to_datetime(gap_df[start_col]),
        "end_time": pd.to_datetime(gap_df[end_col]),
    })

    out = out.dropna()
    out = out[out["end_time"] >= out["start_time"]]

    return out.reset_index(drop=True)


def merge_intervals_fast(intervals_df):
    """
    start_time, end_time を持つ interval DataFrame を高速に merge する。
    """

    if intervals_df is None or len(intervals_df) == 0:
        return pd.DataFrame(columns=["start_time", "end_time"])

    df = intervals_df.dropna().copy()
    df = df.sort_values("start_time").reset_index(drop=True)

    starts = pd.DatetimeIndex(df["start_time"]).to_numpy()
    ends = pd.DatetimeIndex(df["end_time"]).to_numpy()

    order = np.argsort(starts)
    starts = starts[order]
    ends = ends[order]

    # ここは datetime64[ns] のまま np.maximum.accumulate できる
    cum_ends = np.maximum.accumulate(ends)

    # 新しい interval が始まる条件
    # start > 直前までの cumulative end
    new_group = np.empty(len(starts), dtype=bool)
    new_group[0] = True
    new_group[1:] = starts[1:] > cum_ends[:-1]

    group_id = np.cumsum(new_group) - 1

    merged_starts = []
    merged_ends = []

    for gid in np.unique(group_id):
        use = group_id == gid
        merged_starts.append(starts[use][0])
        merged_ends.append(ends[use].max())

    return pd.DataFrame({
        "start_time": pd.to_datetime(merged_starts),
        "end_time": pd.to_datetime(merged_ends),
    })


def make_forbidden_intervals_fast(
    bad_times=None,
    LEPe_gap_ranges_df=None,
    LEPi_gap_ranges_df=None,
    att_gap_ranges_df=None,
):
    """
    bad_times, LEPe gap, LEPi gap をまとめて禁止 interval にする。

    bad_times は point-like interval [t, t] として扱う。
    """

    dfs = []

    if bad_times is not None:
        bad_times_pd = pd.DatetimeIndex(pd.to_datetime(bad_times.values))
        bad_times_pd = bad_times_pd.dropna().sort_values().unique()
        bad_times_pd = pd.DatetimeIndex(bad_times_pd)

        if len(bad_times_pd) > 0:
            dfs.append(pd.DataFrame({
                "start_time": bad_times_pd,
                "end_time": bad_times_pd,
            }))

    df_lepe = gap_df_to_intervals_fast(LEPe_gap_ranges_df)
    df_lepi = gap_df_to_intervals_fast(LEPi_gap_ranges_df)
    df_att  = gap_df_to_intervals_fast(att_gap_ranges_df)

    if len(df_lepe) > 0:
        dfs.append(df_lepe)

    if len(df_lepi) > 0:
        dfs.append(df_lepi)

    if len(df_att) > 0:
        dfs.append(df_att)

    if len(dfs) == 0:
        return pd.DataFrame(columns=["start_time", "end_time"])

    intervals_df = pd.concat(dfs, ignore_index=True)
    intervals_df = merge_intervals_fast(intervals_df)

    return intervals_df

In [ ]:
def make_trimmed_time_ranges_fast(
    selected_times,
    forbidden_intervals_df,
    half_width=pd.Timedelta(minutes=15),
    eps=pd.Timedelta(nanoseconds=1),
):
    """
    selected_times ± half_width の区間を作る。
    ただし forbidden intervals を含まないように、
    各 selected_time の前後で最も近い forbidden interval まで切り詰める。

    Returns
    -------
    time_ranges : list[tuple[pd.Timestamp, pd.Timestamp]]
    """

    selected_times_pd = pd.DatetimeIndex(
        pd.to_datetime(selected_times)
    ).sort_values()

    if len(selected_times_pd) == 0:
        return []

    start0 = selected_times_pd - half_width
    end0 = selected_times_pd + half_width

    if forbidden_intervals_df is None or len(forbidden_intervals_df) == 0:
        return list(zip(start0, end0))

    forbidden_intervals_df = forbidden_intervals_df.sort_values("start_time").reset_index(drop=True)

    f_starts = pd.DatetimeIndex(pd.to_datetime(forbidden_intervals_df["start_time"]))
    f_ends = pd.DatetimeIndex(pd.to_datetime(forbidden_intervals_df["end_time"]))

    # numpy datetime64[ns] として扱う
    t_np = selected_times_pd.to_numpy()
    start_np = start0.to_numpy()
    end_np = end0.to_numpy()

    f_starts_np = f_starts.to_numpy()
    f_ends_np = f_ends.to_numpy()

    # 各 t に対して、t 以下で始まる最後の forbidden interval
    i_prev = np.searchsorted(f_starts_np, t_np, side="right") - 1

    valid_prev = i_prev >= 0

    inside_forbidden = np.zeros(len(t_np), dtype=bool)

    # t が forbidden interval 内に入っているか
    inside_forbidden[valid_prev] = (
        t_np[valid_prev] <= f_ends_np[i_prev[valid_prev]]
    )

    # start 側の切り詰め
    start_trim = start_np.copy()

    use_prev_trim = np.zeros(len(t_np), dtype=bool)
    use_prev_trim[valid_prev] = (
        f_ends_np[i_prev[valid_prev]] > start_np[valid_prev]
    )

    start_trim[use_prev_trim] = (
        f_ends_np[i_prev[use_prev_trim]] + np.timedelta64(eps.value, "ns")
    )

    # next forbidden interval
    # inside でない場合、直後の interval は i_prev + 1
    i_next = i_prev + 1
    valid_next = i_next < len(f_starts_np)

    end_trim = end_np.copy()

    use_next_trim = np.zeros(len(t_np), dtype=bool)
    use_next_trim[valid_next] = (
        f_starts_np[i_next[valid_next]] < end_np[valid_next]
    )

    end_trim[use_next_trim] = (
        f_starts_np[i_next[use_next_trim]] - np.timedelta64(eps.value, "ns")
    )

    valid = (~inside_forbidden) & (end_trim > start_trim)

    start_out = pd.to_datetime(start_trim[valid])
    end_out = pd.to_datetime(end_trim[valid])

    time_ranges = list(zip(start_out, end_out))

    return time_ranges

In [ ]:
def merge_time_ranges(
    time_ranges,
    max_duration=None,
):
    """
    overlapping time ranges を merge する。
    max_duration を指定した場合、merge 後の duration がそれを超える merge はしない。
    """

    if len(time_ranges) == 0:
        return []

    time_ranges = sorted(time_ranges)

    merged = []

    for start, end in time_ranges:
        if not merged:
            merged.append((start, end))
            continue

        prev_start, prev_end = merged[-1]

        is_overlapping = start <= prev_end

        candidate_start = prev_start
        candidate_end = max(prev_end, end)

        if max_duration is None:
            ok_duration = True
        else:
            ok_duration = (candidate_end - candidate_start) <= max_duration

        if is_overlapping and ok_duration:
            merged[-1] = (candidate_start, candidate_end)
        else:
            merged.append((start, end))

    return merged

In [ ]:
import time

t0_clock = time.perf_counter()

# ============================================================
# forbidden intervals を作る
# ============================================================

forbidden_intervals_df = make_forbidden_intervals_fast(
    bad_times=bad_times,
    LEPe_gap_ranges_df=LEPe_omniflux_gap_ranges_df,
    LEPi_gap_ranges_df=LEPi_omniflux_gap_ranges_df,
    att_gap_ranges_df=att_nan_intervals_df
)

print(f"forbidden intervals: {len(forbidden_intervals_df)}")

# ============================================================
# selected_times ±15 min を forbidden interval で切る
# ============================================================

time_ranges = make_trimmed_time_ranges_fast(
    selected_times=selected_times,
    forbidden_intervals_df=forbidden_intervals_df,
    half_width=pd.Timedelta(minutes=15),
    eps=pd.Timedelta(nanoseconds=1),
)

print(f"time_ranges before merge: {len(time_ranges)}")

# ============================================================
# merge
# ============================================================

merged_ranges = merge_time_ranges(
    time_ranges,
    max_duration=None,
    # max_duration=pd.Timedelta(minutes=60),  # 必要ならこちら
)

# ============================================================
# DataFrame 化
# ============================================================

merged_ranges_df = pd.DataFrame(
    merged_ranges,
    columns=["start_time", "end_time"]
)

if len(merged_ranges_df) > 0:
    merged_ranges_df["duration_minutes"] = (
        merged_ranges_df["end_time"] - merged_ranges_df["start_time"]
    ).dt.total_seconds() / 60

    merged_ranges_df = merged_ranges_df[
        merged_ranges_df["duration_minutes"] > 30
    ].reset_index(drop=True)
else:
    merged_ranges_df["duration_minutes"] = []

elapsed = time.perf_counter() - t0_clock

print(f"元の選択時間数: {len(selected_times)}")
print(f"30分超のマージ後時間範囲数: {len(merged_ranges_df)}")
print(f"elapsed: {elapsed:.2f} sec")
print("\n有効なマージ時間範囲:")
print(merged_ranges_df)

# MGF dataのFFT

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
from tqdm.auto import tqdm


def _format_erg_trange(start, end):
    start = pd.to_datetime(start)
    end = pd.to_datetime(end)
    return [
        start.strftime("%Y%m%d/%H:%M:%S"),
        end.strftime("%Y%m%d/%H:%M:%S"),
    ]


def _drop_duplicate_times(da, time_dim="time"):
    times = pd.DatetimeIndex(pd.to_datetime(da[time_dim].values))
    keep = ~times.duplicated(keep="first")
    return da.isel({time_dim: keep})


def load_mgf64_for_ranges(
    ranges_df,
    pad=pd.Timedelta(seconds=2),
    no_update=True,
):
    """
    merged_ranges_df の有効区間だけ MGF 64 Hz data を読み込む。
    pad は FFT 窓端で必要になる余裕分。
    """

    if ranges_df is None or len(ranges_df) == 0:
        raise ValueError("ranges_df が空なので MGF data を読み込めない。")

    B_list = []
    qf_list = []

    ranges = ranges_df.dropna(subset=["start_time", "end_time"]).copy()
    ranges["start_time"] = pd.to_datetime(ranges["start_time"])
    ranges["end_time"] = pd.to_datetime(ranges["end_time"])
    ranges = ranges[ranges["end_time"] > ranges["start_time"]].reset_index(drop=True)

    for i, row in tqdm(ranges.iterrows(), total=len(ranges), desc="Loading MGF 64 Hz"):
        start = row["start_time"] - pad
        end = row["end_time"] + pad
        trange_i = _format_erg_trange(start, end)

        # MGF tplot 変数だけ消して、各区間を独立に読む。
        pt.del_data("erg_mgf_l2_mag_64hz_dsi")
        pt.del_data("erg_mgf_l2_quality_64hz")

        ergpy.mgf(
            trange=trange_i,
            level="l2",
            datatype="64hz",
            get_support_data=True,
            no_update=no_update,
        )

        B_i = psp.get_data("erg_mgf_l2_mag_64hz_dsi", xarray=True)
        qf_i = psp.get_data("erg_mgf_l2_quality_64hz", xarray=True)

        if B_i is None or qf_i is None:
            print(f"skip MGF load failed: {trange_i}")
            continue

        B_i = B_i.sortby("time").sel(time=slice(start, end))
        qf_i = qf_i.sortby("time").sel(time=slice(start, end))

        if B_i.sizes.get("time", 0) == 0 or qf_i.sizes.get("time", 0) == 0:
            print(f"skip empty MGF data: {trange_i}")
            continue

        B_list.append(B_i)
        qf_list.append(qf_i)

        print(f"loaded MGF {i + 1}/{len(ranges)}: {trange_i[0]} - {trange_i[1]}")

    if len(B_list) == 0 or len(qf_list) == 0:
        raise ValueError("有効区間内で MGF data を読み込めなかった。")

    B64_data_dsi = xr.concat(B_list, dim="time").sortby("time")
    B64_data_dsi_quality_flag = xr.concat(qf_list, dim="time").sortby("time")

    B64_data_dsi = _drop_duplicate_times(B64_data_dsi)
    B64_data_dsi_quality_flag = _drop_duplicate_times(B64_data_dsi_quality_flag)

    return B64_data_dsi, B64_data_dsi_quality_flag


B64_data_dsi, B64_data_dsi_quality_flag = load_mgf64_for_ranges(
    merged_ranges_df,
    pad=pd.Timedelta(seconds=2),
    no_update=False,
)

print(B64_data_dsi)
print(B64_data_dsi_quality_flag)


In [ ]:
qf_B, B64 = xr.align(B64_data_dsi_quality_flag[:, 3], B64_data_dsi, join="inner")

B64_data_dsi_qf = xr.where(qf_B <= 21, B64, np.nan)

print(B64_data_dsi_qf)

In [ ]:
B64_data_dsi_qf_total = np.sqrt(xr.dot(B64_data_dsi_qf, B64_data_dsi_qf, dim='v_dim'))

In [ ]:
m_e     = 9.1093837E-31    #[kg]
m_H     = 1.6726219e-27  # kg
m_He    = m_H * 4.
m_O     = m_H * 16.
elementary_charge = 1.60217663E-19  #[A s]

f_cH    = elementary_charge * B64_data_dsi_qf_total*1E-9 / m_H  / 2. / np.pi
f_cHe   = elementary_charge * B64_data_dsi_qf_total*1E-9 / m_He / 2. / np.pi
f_cO    = elementary_charge * B64_data_dsi_qf_total*1E-9 / m_O / 2. / np.pi

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from matplotlib.colors import Normalize
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

path_base_save_plot = f'/mnt/j/statistical_analysis_arase/preanalysis'
path_base_save_plot = Path(path_base_save_plot + f'/PWE-EFD_spec')
os.makedirs(path_base_save_plot, exist_ok=True)

def edges_numeric(arr):
    arr = np.asarray(arr, dtype=float)
    if arr.size < 2:
        return np.concatenate((arr, arr + 1.0))
    d = np.diff(arr) / 2.0
    return np.concatenate(([arr[0] - d[0]], arr[:-1] + d, [arr[-1] + d[-1]]))

vmax = np.log10(1E1)
vmin = np.log10(1e-4)

for idx, row in merged_ranges_df.iterrows():
    start = row["start_time"]
    end = row["end_time"]

    if pd.isna(start) or pd.isna(end):
        continue

    subset_range = efd_spectra_qf.sel(time=slice(start, end))

    if subset_range.sizes.get("time", 0) < 2:
        continue

    times = pd.to_datetime(subset_range.time.values)

    if "spec_bins" in subset_range.coords:
        freqs = subset_range["spec_bins"].values
    elif "v" in subset_range.coords:
        freqs = subset_range["v"].values
    else:
        freqs = np.arange(subset_range.sizes["v_dim"])

    data_range = subset_range.values
    data_range = np.where(
        np.isfinite(data_range) & (data_range > 0),
        data_range,
        np.nan
    )

    logdata_range = np.log10(data_range)

    if np.isnan(logdata_range).all():
        continue

    t_num = mdates.date2num(times.to_pydatetime())
    t_edges = edges_numeric(t_num)
    f_edges = edges_numeric(freqs)

    fig_i, ax_i = plt.subplots(figsize=(12, 4.5))

    pcm_i = ax_i.pcolormesh(
        t_edges,
        f_edges,
        logdata_range.T,
        shading="auto",
        cmap="turbo",
        norm=Normalize(vmin=vmin, vmax=vmax),
    )

    f_cH_plot = f_cH.sel(time=slice(start, end))
    f_cHe_plot = f_cHe.sel(time=slice(start, end))
    f_cO_plot = f_cO.sel(time=slice(start, end))

    ax_i.plot(f_cH_plot.time, f_cH_plot.data, lw=1, c='white', linestyle='solid')
    ax_i.plot(f_cHe_plot.time, f_cHe_plot.data, lw=1, c='magenta', linestyle='solid')
    ax_i.plot(f_cO_plot.time, f_cO_plot.data, lw=1, c='yellow', linestyle='solid')

    ax_i.set_xlabel("Time")
    ax_i.xaxis_date()
    ax_i.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    ax_i.set_ylabel("Frequency [Hz]")
    ax_i.set_title(
        f"PWE-EFD spec ({start:%Y-%m-%d %H:%M:%S} - {end:%Y-%m-%d %H:%M:%S})"
    )
    ax_i.set_yscale("log")
    ax_i.set_ylim(3, 32)
    ax_i.set_xlim(start.to_pydatetime(), end.to_pydatetime())

    fig_i.colorbar(pcm_i, ax=ax_i, label=r"log10 PSD [$\mathrm{(mV/m)}^2/\mathrm{Hz}$]")
    plt.tight_layout()

    # =========================
    # save figure
    # =========================

    # start の月・日で階層フォルダを作る
    save_dir = path_base_save_plot / f"{start:%Y}" / f"{start:%m}"
    save_dir.mkdir(parents=True, exist_ok=True)

    filename = (
        f"efd_spectra_qf_"
        f"{start:%Y%m%d_%H%M%S}_to_{end:%Y%m%d_%H%M%S}.png"
    )

    save_path = save_dir / filename

    fig_i.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.close(fig_i)

    print(f"saved: {save_path}")

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr


def _estimate_sampling_rate_from_time(time_values):
    """
    datetime64 time coordinate から sampling rate を推定する。
    """
    time = pd.DatetimeIndex(pd.to_datetime(time_values))
    t_sec = (time - time[0]).total_seconds().to_numpy()

    dt_arr = np.diff(t_sec)
    dt_arr = dt_arr[np.isfinite(dt_arr) & (dt_arr > 0)]

    if len(dt_arr) == 0:
        raise ValueError("time 軸から dt を推定できない。")

    dt = np.median(dt_arr)
    fs = 1.0 / dt

    return fs, dt


def _one_sided_psd(x, fs, use_hann=True, remove_mean=True):
    """
    1D waveform から one-sided PSD を計算する。

    入力 x の単位が nT なら、出力 PSD は nT^2/Hz。
    """

    x = np.asarray(x, dtype=float)

    if np.any(~np.isfinite(x)):
        raise ValueError("x に NaN/Inf が含まれている。")

    if remove_mean:
        x = x - np.mean(x)

    N = len(x)

    if use_hann:
        window = np.hanning(N)
    else:
        window = np.ones(N)

    xw = x * window

    fft_vals = np.fft.rfft(xw)
    freq = np.fft.rfftfreq(N, d=1.0 / fs)

    psd = np.abs(fft_vals) ** 2 / (fs * np.sum(window ** 2))

    # one-sided correction
    if N % 2 == 0:
        # even N: DC と Nyquist 以外を2倍
        psd[1:-1] *= 2.0
    else:
        # odd N: DC 以外を2倍
        psd[1:] *= 2.0

    return freq, psd


def _bin_psd_to_1hz_bins(freq, psd, spec_bins=np.arange(3, 33), bin_width=1.0):
    """
    raw FFT PSD を 1 Hz ごとの spec_bins にまとめる。

    spec_bins は bin center として扱う。
    spec_bins=3 は 2.5--3.5 Hz の平均 PSD。
    spec_bins=32 は 31.5--32.5 Hz だが、Nyquist により実質 31.5--32 Hz。
    """

    freq = np.asarray(freq)
    psd = np.asarray(psd)

    out = np.full(len(spec_bins), np.nan)

    for i, fc in enumerate(spec_bins):
        f0 = fc - bin_width / 2.0
        f1 = fc + bin_width / 2.0

        if i == len(spec_bins) - 1:
            use = (freq >= f0) & (freq <= f1)
        else:
            use = (freq >= f0) & (freq < f1)

        if np.any(use):
            out[i] = np.nanmean(psd[use])

    return out

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm


def make_mgf64_spectra_like_efd(
    da_B,
    target_times=None,
    spec_bins=np.arange(3, 33),
    window_sec=1.0,
    step_sec=1.0,
    min_valid_fraction=1.0,
    gap_factor=1.5,
    use_hann=True,
    remove_mean=True,
    name="B64_spectra_qf",
    show_progress=True,
    n_workers=1,
):
    """
    MGF 64 Hz total magnetic field から EFD spec と同様の
    time x spec_bins スペクトルプロダクトを作る。

    Parameters
    ----------
    da_B : xr.DataArray
        time 軸を持つ 1D DataArray。
        単位は nT を想定。
        quality flag 不良点は NaN になっているものとする。
    target_times : array-like or None
        出力する spectrum の中心時刻。
        EFD と完全に同じ time grid にしたい場合は
        target_times=efd_spectra_qf.time.values
        とする。
        None の場合は step_sec ごとの時刻を自動生成する。
    spec_bins : array-like
        固定 frequency bins。デフォルトは 3--32 Hz。
    window_sec : float
        FFT 窓長。64 Hz sampling で 1秒なら df=1 Hz。
    step_sec : float
        target_times=None の場合の出力 cadence。
    min_valid_fraction : float
        各窓内で必要な有限値の割合。
        1.0 なら NaN が1点でもあれば、その窓は NaN spectrum。
    gap_factor : float
        窓内の time gap が通常 dt の gap_factor 倍を超えたら無効化。
    use_hann : bool
        Hann window を使うか。
    remove_mean : bool
        各窓で mean を除去するか。
    name : str
        出力 DataArray 名。
    show_progress : bool
        True の場合、FFT計算の進捗バーを表示する。
    n_workers : int
        1なら直列処理。2以上なら ThreadPoolExecutor で target_times を並列処理する。

    Returns
    -------
    spectra : xr.DataArray
        dims: time, spec_bins
        units: nT^2/Hz
    """

    da_B = da_B.sortby("time")

    time = pd.DatetimeIndex(pd.to_datetime(da_B.time.values))
    values = da_B.values.astype(float)
    spec_bins = np.asarray(spec_bins)

    fs, dt = _estimate_sampling_rate_from_time(time)

    nperseg = int(round(window_sec * fs))

    if nperseg < 2:
        raise ValueError("window_sec が短すぎる。")

    # 1秒窓、64 Hzなら nperseg = 64
    actual_window_sec = nperseg / fs
    df = fs / nperseg
    nyq = fs / 2.0

    print(f"Estimated fs        : {fs:.6f} Hz")
    print(f"Estimated dt        : {dt:.6f} sec")
    print(f"nperseg             : {nperseg}")
    print(f"actual window length: {actual_window_sec:.6f} sec")
    print(f"df                  : {df:.6f} Hz")
    print(f"Nyquist             : {nyq:.6f} Hz")
    print(f"target_times        : {0 if target_times is None else len(target_times)}")
    print(f"n_workers           : {n_workers}")

    if np.max(spec_bins) > nyq + 1e-6:
        print(
            f"Warning: spec_bins max = {np.max(spec_bins)} Hz exceeds Nyquist = {nyq:.3f} Hz"
        )

    half = nperseg // 2

    # target_times を作る
    if target_times is None:
        t0 = time[0] + pd.Timedelta(seconds=actual_window_sec / 2.0)
        t1 = time[-1] - pd.Timedelta(seconds=actual_window_sec / 2.0)

        target_times = pd.date_range(
            start=t0,
            end=t1,
            freq=pd.to_timedelta(step_sec, unit="s"),
        )
    else:
        target_times = pd.DatetimeIndex(pd.to_datetime(target_times))

    spectra = np.full((len(target_times), len(spec_bins)), np.nan)
    valid_fraction_arr = np.full(len(target_times), np.nan)
    n_valid_arr = np.full(len(target_times), 0)
    is_valid_window = np.full(len(target_times), False)

    time_ns = time.asi8

    def process_one(it):
        tc = target_times[it]
        tc_ns = pd.Timestamp(tc).value

        # 中心時刻に最も近い index
        ic = np.searchsorted(time_ns, tc_ns)

        # even nperseg の場合、中心の取り方は少し任意性がある。
        # ここでは ic を中心近傍として nperseg 点を取る。
        i0 = ic - half
        i1 = i0 + nperseg

        empty_spectrum = np.full(len(spec_bins), np.nan)

        if i0 < 0 or i1 > len(values):
            return it, empty_spectrum, np.nan, 0, False

        x = values[i0:i1]
        t_win = time[i0:i1]

        if len(x) != nperseg:
            return it, empty_spectrum, np.nan, 0, False

        # time gap check
        t_win_sec = (t_win - t_win[0]).total_seconds().to_numpy()
        dt_win = np.diff(t_win_sec)

        if len(dt_win) == 0:
            return it, empty_spectrum, np.nan, 0, False

        if np.nanmax(dt_win) > gap_factor * dt:
            return it, empty_spectrum, np.nan, 0, False

        finite = np.isfinite(x)
        valid_fraction = finite.sum() / len(x)
        n_valid = int(finite.sum())

        if valid_fraction < min_valid_fraction:
            return it, empty_spectrum, valid_fraction, n_valid, False

        # 今回は quality flag 不良点を補間しない方針。
        # min_valid_fraction=1.0 なら、ここに来る時点で NaN はない。
        if np.any(~finite):
            return it, empty_spectrum, valid_fraction, n_valid, False

        freq, psd = _one_sided_psd(
            x,
            fs=fs,
            use_hann=use_hann,
            remove_mean=remove_mean,
        )

        spectrum = _bin_psd_to_1hz_bins(
            freq=freq,
            psd=psd,
            spec_bins=spec_bins,
            bin_width=1.0,
        )

        return it, spectrum, valid_fraction, n_valid, True

    indices = range(len(target_times))

    if n_workers is None or n_workers <= 1:
        iterator = indices
        if show_progress:
            iterator = tqdm(iterator, total=len(target_times), desc="MGF FFT")

        for result in iterator:
            it, spectrum, valid_fraction, n_valid, ok = process_one(result)
            spectra[it, :] = spectrum
            valid_fraction_arr[it] = valid_fraction
            n_valid_arr[it] = n_valid
            is_valid_window[it] = ok
    else:
        with ThreadPoolExecutor(max_workers=int(n_workers)) as executor:
            iterator = executor.map(process_one, indices)
            if show_progress:
                iterator = tqdm(iterator, total=len(target_times), desc="MGF FFT")

            for it, spectrum, valid_fraction, n_valid, ok in iterator:
                spectra[it, :] = spectrum
                valid_fraction_arr[it] = valid_fraction
                n_valid_arr[it] = n_valid
                is_valid_window[it] = ok

    spectra_da = xr.DataArray(
        spectra,
        coords={
            "time": target_times,
            "spec_bins": spec_bins,
            "valid_fraction": ("time", valid_fraction_arr),
            "n_valid": ("time", n_valid_arr),
            "is_valid_window": ("time", is_valid_window),
        },
        dims=("time", "spec_bins"),
        name=name,
        attrs={
            "units": "nT^2/Hz",
            "input_units": "nT",
            "method": "sliding-window one-sided FFT PSD",
            "window": "hann" if use_hann else "boxcar",
            "remove_mean": bool(remove_mean),
            "sampling_rate_Hz": float(fs),
            "dt_sec": float(dt),
            "nperseg": int(nperseg),
            "window_sec": float(actual_window_sec),
            "df_Hz": float(df),
            "nyquist_Hz": float(nyq),
            "spec_bins": "frequency bin centers [Hz]",
            "bin_width_Hz": 1.0,
            "min_valid_fraction": float(min_valid_fraction),
            "n_workers": int(n_workers) if n_workers is not None else 1,
            "quality_flag_handling": "NaN samples are not interpolated; windows with insufficient valid samples are set to NaN.",
        },
    )

    return spectra_da


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import xarray as xr


def make_times_in_ranges(time_values, ranges_df):
    """
    time_values のうち、ranges_df の start_time/end_time に含まれるものだけ返す。
    """

    times = pd.DatetimeIndex(pd.to_datetime(time_values))

    if ranges_df is None or len(ranges_df) == 0:
        return times.values

    use = np.zeros(len(times), dtype=bool)

    ranges = ranges_df.dropna(subset=["start_time", "end_time"]).copy()
    ranges["start_time"] = pd.to_datetime(ranges["start_time"])
    ranges["end_time"] = pd.to_datetime(ranges["end_time"])

    for _, row in ranges.iterrows():
        use |= (times >= row["start_time"]) & (times <= row["end_time"])

    return times[use].values


def make_b64_spectra_save_path(
    ranges_df,
    time_range,
    base_dir=Path("/mnt/j/observation_data/statistical_analysis_arase/Arase_analysis_save_data/B64_spectra"),
):
    """
    有効区間限定 B64_spectra_qf の保存パスを一意に作る。
    """

    base_dir = Path(base_dir)
    base_dir.mkdir(parents=True, exist_ok=True)

    if ranges_df is not None and len(ranges_df) > 0:
        t0 = pd.to_datetime(ranges_df["start_time"].min())
        t1 = pd.to_datetime(ranges_df["end_time"].max())
    else:
        t0 = pd.to_datetime(time_range[0])
        t1 = pd.to_datetime(time_range[-1])

    filename = (
        f"B64_spectra_qf_valid_ranges_"
        f"{t0:%Y%m%d_%H%M%S}_to_{t1:%Y%m%d_%H%M%S}.nc"
    )

    return base_dir / filename


mgf_target_times = make_times_in_ranges(
    efd_spectra_qf.time.values,
    merged_ranges_df,
)

B64_spectra_save_path = make_b64_spectra_save_path(
    merged_ranges_df,
    time_range,
)

print(f"EFD time grid 全体: {efd_spectra_qf.sizes.get('time', 0)}")
print(f"MGF FFT target_times: {len(mgf_target_times)}")
print(f"B64 spectra save path: {B64_spectra_save_path}")


In [ ]:
# ============================================================
# Load cached B64_spectra_qf if available; otherwise compute and save.
# ============================================================

force_recompute_B64_spectra = False

if B64_spectra_save_path.exists() and not force_recompute_B64_spectra:
    ds_loaded = xr.open_dataset(B64_spectra_save_path)
    B64_spectra_qf = ds_loaded["B64_spectra_qf"]
    print(f"loaded cached B64_spectra_qf: {B64_spectra_save_path}")

else:
    if len(mgf_target_times) == 0:
        raise ValueError("mgf_target_times が空です。merged_ranges_df または efd_spectra_qf を確認してください。")

    B64_spectra_qf = make_mgf64_spectra_like_efd(
        B64_data_dsi_qf_total,
        target_times=mgf_target_times,
        spec_bins=np.arange(3, 33),
        window_sec=1.0,
        min_valid_fraction=1.0,
        use_hann=True,
        remove_mean=True,
        show_progress=True,
        n_workers=16,
    )

    B64_spectra_qf.attrs.update({
        "long_name": "MGF 64 Hz total magnetic field spectrum",
        "units": "nT^2/Hz",
        "input_data": "B64_data_dsi_qf_total",
        "quality_flag": "Bad-quality samples were set to NaN before FFT.",
        "frequency_bins": "1 Hz bins from 3 to 32 Hz",
        "analysis_ranges": "Only merged_ranges_df intervals were loaded and transformed.",
    })

    ds_B64_spectra_qf = B64_spectra_qf.to_dataset(name="B64_spectra_qf")

    # NetCDFで扱いやすい属性に整える。
    for key, val in list(ds_B64_spectra_qf.attrs.items()):
        if isinstance(val, (bool, np.bool_)):
            ds_B64_spectra_qf.attrs[key] = str(val)

    for var in ds_B64_spectra_qf.variables:
        for key, val in list(ds_B64_spectra_qf[var].attrs.items()):
            if isinstance(val, (bool, np.bool_)):
                ds_B64_spectra_qf[var].attrs[key] = str(val)

    ds_B64_spectra_qf.to_netcdf(B64_spectra_save_path)
    print(f"computed and saved B64_spectra_qf: {B64_spectra_save_path}")

print(B64_spectra_qf)


In [ ]:
# 確認用: B64_spectra_qf の時間範囲と有効窓数
if B64_spectra_qf.sizes.get("time", 0) > 0:
    print(f"B64_spectra_qf time start: {pd.to_datetime(B64_spectra_qf.time.values[0])}")
    print(f"B64_spectra_qf time end  : {pd.to_datetime(B64_spectra_qf.time.values[-1])}")
    if "is_valid_window" in B64_spectra_qf.coords:
        print(f"valid FFT windows: {int(B64_spectra_qf.is_valid_window.sum().values)}")
else:
    print("B64_spectra_qf is empty")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.colors import Normalize
from pathlib import Path


def edges_numeric(arr):
    arr = np.asarray(arr, dtype=float)
    if arr.size < 2:
        return np.concatenate((arr, arr + 1.0))
    d = np.diff(arr) / 2.0
    return np.concatenate(([arr[0] - d[0]], arr[:-1] + d, [arr[-1] + d[-1]]))


# ============================================================
# save base path
# ============================================================

path_base_save_plot = f'/mnt/j/statistical_analysis_arase/preanalysis'
path_base_save_plot = Path(path_base_save_plot + f'/MGF_spec')
os.makedirs(path_base_save_plot, exist_ok=True)



# ============================================================
# plot settings
# ============================================================

vmin = -4
vmax = -1

for idx, row in merged_ranges_df.iterrows():

    start = pd.to_datetime(row["start_time"])
    end   = pd.to_datetime(row["end_time"])

    if pd.isna(start) or pd.isna(end):
        continue

    f_cH_plot = f_cH.sel(time=slice(start, end))
    f_cHe_plot = f_cHe.sel(time=slice(start, end))
    f_cO_plot = f_cO.sel(time=slice(start, end))

    # xarray sel 用
    B64_spectra_qf_analysis = B64_spectra_qf.sel(
        time=slice(start, end)
    )

    # time が少なすぎる場合は skip
    if B64_spectra_qf_analysis.sizes.get("time", 0) < 2:
        print(f"skip: too few time points: {start} - {end}")
        continue

    data = B64_spectra_qf_analysis.values
    data = np.where(np.isfinite(data) & (data > 0), data, np.nan)

    if np.isnan(data).all():
        print(f"skip: all NaN: {start} - {end}")
        continue

    logdata = np.log10(data)

    times = pd.DatetimeIndex(pd.to_datetime(B64_spectra_qf_analysis.time.values))
    t_num = mdates.date2num(times.to_pydatetime())
    t_edges = edges_numeric(t_num)

    freqs = B64_spectra_qf_analysis.spec_bins.values
    f_edges = edges_numeric(freqs)

    fig, ax = plt.subplots(figsize=(12, 4.5))

    pcm = ax.pcolormesh(
        t_edges,
        f_edges,
        logdata.T,
        shading="auto",
        cmap="turbo",
        norm=Normalize(vmin=vmin, vmax=vmax),
    )

    ax.plot(f_cH_plot.time, f_cH_plot.data, lw=1, c='white', linestyle='solid')
    ax.plot(f_cHe_plot.time, f_cHe_plot.data, lw=1, c='magenta', linestyle='solid')
    ax.plot(f_cO_plot.time, f_cO_plot.data, lw=1, c='yellow', linestyle='solid')

    ax.xaxis_date()
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))

    ax.set_yscale("log")
    ax.set_ylim(3, 32)

    ax.set_xlim(start.to_pydatetime(), end.to_pydatetime())

    ax.set_xlabel("Time")
    ax.set_ylabel("Frequency [Hz]")
    ax.set_title(
        f"MGF 64hz total spec "
        f"({start:%Y-%m-%d %H:%M:%S} - {end:%Y-%m-%d %H:%M:%S})"
    )

    fig.colorbar(
        pcm,
        ax=ax,
        label=r"log10 PSD [$\mathrm{nT}^2/\mathrm{Hz}$]"
    )

    plt.tight_layout()

    # ========================================================
    # save
    # ========================================================

    save_dir = path_base_save_plot / f"{start:%Y}" / f"{start:%m}"
    save_dir.mkdir(parents=True, exist_ok=True)

    filename = (
        f"MGF_B64_spectra_"
        f"{start:%Y%m%d_%H%M%S}_to_{end:%Y%m%d_%H%M%S}.png"
    )

    save_path = save_dir / filename

    fig.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.close(fig)

    print(f"saved: {save_path}")

# 各selected_timeでのMGF PSDの勾配を確認

In [ ]:
import numpy as np
import pandas as pd
from tqdm.contrib.concurrent import thread_map


def quantify_decreasing_psd_array(f, P):
    """
    1時刻の MGF PSD spectrum が 3--32 Hz で概ね単調減少かを定量化する。
    xarray を使わない高速版。
    """

    f = np.asarray(f, dtype=float)
    P = np.asarray(P, dtype=float)

    use = np.isfinite(f) & np.isfinite(P) & (P > 0)

    f = f[use]
    P = P[use]

    if len(f) < 3:
        return {
            "n_points": len(f),
            "alpha": np.nan,
            "intercept": np.nan,
            "r2_loglog": np.nan,
            "rmse_log10": np.nan,
            "spearman_rho": np.nan,
            "adjacent_decrease_fraction": np.nan,
            "positive_local_slope_fraction": np.nan,
            "median_local_alpha": np.nan,
            "log10_drop_low_to_high": np.nan,
            "monotonic_decrease_score": np.nan,
        }

    order = np.argsort(f)
    f = f[order]
    P = P[order]

    logf = np.log10(f)
    logP = np.log10(P)

    # log10(PSD) = intercept + slope * log10(f)
    # PSD が f^(-alpha) 型で減少する場合 alpha > 0。
    slope, intercept = np.polyfit(logf, logP, 1)
    alpha = -slope

    fit = intercept + slope * logf
    resid = logP - fit

    ss_res = np.sum(resid**2)
    ss_tot = np.sum((logP - np.mean(logP))**2)

    r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else np.nan
    rmse_log10 = np.sqrt(np.mean(resid**2))

    spearman_rho = pd.Series(logf).corr(
        pd.Series(logP),
        method="spearman",
    )

    dlogP = np.diff(logP)
    dlogf = np.diff(logf)

    adjacent_decrease_fraction = np.mean(dlogP < 0)

    local_alpha = -dlogP / dlogf

    positive_local_slope_fraction = np.mean(local_alpha > 0)
    median_local_alpha = np.nanmedian(local_alpha)

    log10_drop_low_to_high = logP[0] - logP[-1]

    decreasing_variation = np.sum(np.maximum(-dlogP, 0))
    total_variation = np.sum(np.abs(dlogP))

    if total_variation > 0:
        monotonic_decrease_score = decreasing_variation / total_variation
    else:
        monotonic_decrease_score = np.nan

    return {
        "n_points": len(f),
        "alpha": alpha,
        "intercept": intercept,
        "r2_loglog": r2,
        "rmse_log10": rmse_log10,
        "spearman_rho": spearman_rho,
        "adjacent_decrease_fraction": adjacent_decrease_fraction,
        "positive_local_slope_fraction": positive_local_slope_fraction,
        "median_local_alpha": median_local_alpha,
        "log10_drop_low_to_high": log10_drop_low_to_high,
        "monotonic_decrease_score": monotonic_decrease_score,
    }


def _check_threshold(value, threshold, mode):
    """
    threshold が None なら条件を課さない。
    mode は 'gt' or 'lt'。
    """

    if threshold is None:
        return True

    if not np.isfinite(value):
        return False

    if mode == "gt":
        return value > threshold
    if mode == "lt":
        return value < threshold

    raise ValueError("mode must be 'gt' or 'lt'")


def make_mask_in_ranges(time_values, ranges_df):
    """
    time_values が ranges_df の start_time--end_time 内にあるかを返す。
    """

    times = pd.DatetimeIndex(pd.to_datetime(time_values))
    use = np.zeros(len(times), dtype=bool)

    if ranges_df is None or len(ranges_df) == 0:
        return use

    ranges = ranges_df.dropna(subset=["start_time", "end_time"]).copy()
    ranges["start_time"] = pd.to_datetime(ranges["start_time"])
    ranges["end_time"] = pd.to_datetime(ranges["end_time"])

    for _, row in ranges.iterrows():
        use |= (times >= row["start_time"]) & (times <= row["end_time"])

    return use


In [ ]:
def make_monotonic_selected_times_parallel(
    B64_spectra_qf,
    selected_times,
    selected_fracs=None,
    source_indices=None,
    freq_dim="spec_bins",
    fmin=3,
    fmax=32,
    alpha_min=0.0,
    alpha_max=None,
    r2_min=0.5,
    spearman_max=-0.7,
    adjacent_decrease_fraction_min=0.65,
    monotonic_decrease_score_min=0.7,
    n_workers=8,
    desc="Checking monotonic B PSD",
    nearest_tolerance=None,
):
    """
    selected_times のうち、B PSD が概ね単調減少する時刻だけを抽出する。

    B64_spectra_qf は前段で merged_ranges_df 内の target_times だけに限定して作るため、
    ここでは selected_times を B64_spectra_qf.time に照合して評価する。
    threshold に None を指定した場合、その条件は使わない。
    """

    da = B64_spectra_qf.transpose("time", freq_dim)
    da = da.sel({freq_dim: slice(fmin, fmax)})

    all_times = pd.DatetimeIndex(pd.to_datetime(da.time.values))
    selected_times_pd = pd.DatetimeIndex(pd.to_datetime(selected_times))

    if source_indices is None:
        source_indices = np.arange(len(selected_times_pd))
    else:
        source_indices = np.asarray(source_indices)

    freqs = da[freq_dim].values.astype(float)
    spectra_values = da.values.astype(float)

    if nearest_tolerance is None:
        time_indexer = all_times.get_indexer(selected_times_pd)
    else:
        time_indexer = all_times.get_indexer(
            selected_times_pd,
            method="nearest",
            tolerance=pd.Timedelta(nearest_tolerance),
        )

    def process_one(i):
        t = selected_times_pd[i]
        idx = time_indexer[i]

        if idx < 0:
            return {
                "source_index": int(source_indices[i]),
                "time": t,
                "matched_time": pd.NaT,
                "n_points": np.nan,
                "alpha": np.nan,
                "intercept": np.nan,
                "r2_loglog": np.nan,
                "rmse_log10": np.nan,
                "spearman_rho": np.nan,
                "adjacent_decrease_fraction": np.nan,
                "positive_local_slope_fraction": np.nan,
                "median_local_alpha": np.nan,
                "log10_drop_low_to_high": np.nan,
                "monotonic_decrease_score": np.nan,
                "is_monotonic_decreasing": False,
                "reject_reason": "no_matching_B64_spectrum",
            }

        P = spectra_values[idx, :]

        metrics = quantify_decreasing_psd_array(
            f=freqs,
            P=P,
        )

        alpha = metrics["alpha"]
        r2 = metrics["r2_loglog"]
        rho = metrics["spearman_rho"]
        dec_frac = metrics["adjacent_decrease_fraction"]
        mono_score = metrics["monotonic_decrease_score"]

        ok = (
            np.isfinite(alpha)
            and np.isfinite(r2)
            and np.isfinite(rho)
            and np.isfinite(dec_frac)
            and np.isfinite(mono_score)
            and _check_threshold(alpha, alpha_min, "gt")
            and _check_threshold(alpha, alpha_max, "lt")
            and _check_threshold(r2, r2_min, "gt")
            and _check_threshold(rho, spearman_max, "lt")
            and _check_threshold(dec_frac, adjacent_decrease_fraction_min, "gt")
            and _check_threshold(mono_score, monotonic_decrease_score_min, "gt")
        )

        row = dict(metrics)
        row["source_index"] = int(source_indices[i])
        row["time"] = t
        row["matched_time"] = all_times[idx]
        row["is_monotonic_decreasing"] = bool(ok)
        row["reject_reason"] = "" if ok else "threshold_not_met"

        return row

    rows = thread_map(
        process_one,
        range(len(selected_times_pd)),
        max_workers=n_workers,
        desc=desc,
    )

    metrics_df = pd.DataFrame(rows).set_index("time")

    mask_mono = metrics_df["is_monotonic_decreasing"].values.astype(bool)

    selected_times_mono = np.asarray(selected_times)[mask_mono]

    if selected_fracs is not None:
        selected_fracs_mono = np.asarray(selected_fracs)[mask_mono]
    else:
        selected_fracs_mono = None

    return selected_times_mono, selected_fracs_mono, metrics_df, mask_mono


In [ ]:
# ============================================================
# 前段の有効時間範囲に対応する selected_times だけ MGF PSD 勾配判定へ渡す
# ============================================================

selected_times = np.asarray(selected_times)
selected_fracs = np.asarray(selected_fracs)

mask_selected_in_merged_ranges = make_mask_in_ranges(
    selected_times,
    merged_ranges_df,
)

selected_times_for_B_check = selected_times[mask_selected_in_merged_ranges]
selected_fracs_for_B_check = selected_fracs[mask_selected_in_merged_ranges]
source_indices_for_B_check = np.where(mask_selected_in_merged_ranges)[0]

print(f"元の selected_times 数: {len(selected_times)}")
print(f"merged_ranges_df 内の selected_times 数: {len(selected_times_for_B_check)}")
print(f"B64_spectra_qf time 数: {B64_spectra_qf.sizes.get('time', 0)}")

selected_times_mono_in_ranges, selected_fracs_mono_in_ranges, B64_mono_metrics_df, mask_mono_in_ranges = make_monotonic_selected_times_parallel(
    B64_spectra_qf=B64_spectra_qf,
    selected_times=selected_times_for_B_check,
    selected_fracs=selected_fracs_for_B_check,
    source_indices=source_indices_for_B_check,
    freq_dim="spec_bins",
    fmin=3,
    fmax=32,

    alpha_min=0,
    alpha_max=None,
    r2_min=0.5,
    spearman_max=-0.7,
    adjacent_decrease_fraction_min=None,
    monotonic_decrease_score_min=0.5,

    n_workers=16,
    nearest_tolerance=None,
)

# mask_mono は元の selected_times と同じ長さに戻しておく。
mask_mono = np.zeros(len(selected_times), dtype=bool)
mask_mono[source_indices_for_B_check] = mask_mono_in_ranges

selected_times_mono = selected_times[mask_mono]
selected_fracs_mono = selected_fracs[mask_mono]

print(f"B PSD が概ね単調減少する selected_times 数: {len(selected_times_mono)}")
print(f"元の selected_times に対する残存率: {len(selected_times_mono) / len(selected_times):.3f}")
print(f"merged_ranges_df 内 selected_times に対する残存率: {len(selected_times_mono) / len(selected_times_for_B_check):.3f}")

B64_mono_metrics_df


# Bzの角度の設定

Breneman+ 2022のRBSPの設定に従い、磁場とspin planeの角度が15°以上の時に有効とする。

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr


def make_EdotB0_geometry_mask(
    B_da,
    angle_min_deg=15.0,
    time_dim="time",
    comp_dim=None,
):
    """
    B_da: xarray.DataArray
        shape = (time, 3) を想定。
        成分順は [Bx, By, Bz]。

    angle_min_deg:
        磁場が spin plane / DSI xy plane から何度以上離れていれば有効とみなすか。
        RBSP 的には 15 deg が一つの目安。

    returns:
        xr.Dataset containing:
            B_spinplane_angle_deg
            Ez_amp_factor
            bad_EdotB0_geometry
    """

    if comp_dim is None:
        comp_dims = [d for d in B_da.dims if d != time_dim]
        if len(comp_dims) != 1:
            raise ValueError("component dimension を特定できないので comp_dim を指定してほしい。")
        comp_dim = comp_dims[0]

    Bx = B_da.isel({comp_dim: 0})
    By = B_da.isel({comp_dim: 1})
    Bz = B_da.isel({comp_dim: 2})

    Bxy = np.sqrt(Bx**2 + By**2)
    Bmag = np.sqrt(Bx**2 + By**2 + Bz**2)

    theta_B_spin = np.degrees(np.arctan2(np.abs(Bz), Bxy))

    Ez_amp_factor = xr.where(
        np.abs(Bz) > 0,
        Bxy / np.abs(Bz),
        np.inf,
    )

    bad_geom = (
        ~np.isfinite(theta_B_spin)
        | ~np.isfinite(Ez_amp_factor)
        | ~np.isfinite(Bmag)
        | (Bmag <= 0)
        | (theta_B_spin < angle_min_deg)
    )

    out = xr.Dataset({
        "B_spinplane_angle_deg": theta_B_spin,
        "Ez_amp_factor": Ez_amp_factor,
        "bad_EdotB0_geometry": bad_geom,
    })

    return out

In [ ]:
geom = make_EdotB0_geometry_mask(
    B64_data_dsi_qf,
    angle_min_deg=15.0,
)

bad_mask_spin_angle_raw = geom["bad_EdotB0_geometry"]

# B64_data_dsi_qf は FFT 端のため pad を含んでいるので、判定に使う範囲は merged_ranges_df 内に戻す。
mask_B64_in_merged_ranges = xr.DataArray(
    make_mask_in_ranges(B64_data_dsi_qf["time"].values, merged_ranges_df),
    coords={"time": B64_data_dsi_qf["time"]},
    dims=("time",),
)

bad_mask_spin_angle = bad_mask_spin_angle_raw & mask_B64_in_merged_ranges
bad_times_spin_angle = B64_data_dsi_qf["time"].where(bad_mask_spin_angle, drop=True)

n_in_ranges = int(mask_B64_in_merged_ranges.sum().values)
n_bad_spin = int(bad_mask_spin_angle.sum().values)
bad_fraction_spin = n_bad_spin / n_in_ranges if n_in_ranges > 0 else np.nan

print(f"B64 samples in merged_ranges_df: {n_in_ranges}")
print(f"bad spin-angle samples: {n_bad_spin}")
print(f"bad fraction in merged_ranges_df = {bad_fraction_spin:.4f}")
print(bad_times_spin_angle)


# EFD波形データの有効時間の確認

In [ ]:
from tqdm.auto import tqdm


def make_efd64_quality_save_path(
    ranges_df,
    time_range,
    base_dir=Path("/mnt/j/observation_data/statistical_analysis_arase/Arase_analysis_save_data/E64_quality"),
):
    """
    有効区間限定 E64 quality flag の保存パスを一意に作る。
    """

    base_dir = Path(base_dir)
    base_dir.mkdir(parents=True, exist_ok=True)

    if ranges_df is not None and len(ranges_df) > 0:
        t0 = pd.to_datetime(ranges_df["start_time"].min())
        t1 = pd.to_datetime(ranges_df["end_time"].max())
    else:
        t0 = pd.to_datetime(time_range[0])
        t1 = pd.to_datetime(time_range[-1])

    filename = (
        f"E64_quality_valid_ranges_"
        f"{t0:%Y%m%d_%H%M%S}_to_{t1:%Y%m%d_%H%M%S}.nc"
    )

    return base_dir / filename


def load_efd64_quality_for_ranges(
    ranges_df,
    pad=pd.Timedelta(seconds=2),
    no_update=True,
    use_cache=True,
    force_reload=False,
    save_path=None,
):
    """
    merged_ranges_df の有効区間だけ EFD 64 Hz waveform quality flag を読み込む。
    pad は後段で時間差分や窓端を扱うときの余裕分。
    use_cache=True の場合、保存済み NetCDF があれば読み込んで pyspedas 読み込みを省略する。
    """

    if ranges_df is None or len(ranges_df) == 0:
        raise ValueError("ranges_df が空なので EFD 64 Hz quality flag を読み込めない。")

    if save_path is not None:
        save_path = Path(save_path)

    if use_cache and save_path is not None and save_path.exists() and not force_reload:
        try:
            ds_loaded = xr.open_dataset(save_path)
            E64_data_dsi_quality_flag = ds_loaded["E64_data_dsi_quality_flag"]
            print(f"loaded cached EFD 64 Hz quality flag: {save_path}")
            return E64_data_dsi_quality_flag
        except Exception as err:
            print(f"failed to load cached EFD 64 Hz quality flag; recompute: {save_path} ({err})")
            try:
                save_path.unlink()
            except OSError:
                pass

    qf_list = []

    ranges = ranges_df.dropna(subset=["start_time", "end_time"]).copy()
    ranges["start_time"] = pd.to_datetime(ranges["start_time"])
    ranges["end_time"] = pd.to_datetime(ranges["end_time"])
    ranges = ranges[ranges["end_time"] > ranges["start_time"]].reset_index(drop=True)

    for i, row in tqdm(ranges.iterrows(), total=len(ranges), desc="Loading EFD 64 Hz quality"):
        start = row["start_time"] - pad
        end = row["end_time"] + pad
        trange_i = _format_erg_trange(start, end)

        pt.del_data("erg_pwe_efd_l2_E64Hz_dsi_quality_flag")

        try:
            ergpy.pwe_efd(
                trange=trange_i,
                level="l2",
                datatype="64",
                coord="dsi",
                get_support_data=True,
                no_update=no_update,
            )
        except ValueError as err:
            # ergpy.pwe_efd may fail inside ylim calculation when the clipped data is empty.
            if "zero-size array" in str(err):
                print(f"skip empty EFD 64 Hz load range: {trange_i} ({err})")
                continue
            raise

        qf_i = psp.get_data("erg_pwe_efd_l2_E64Hz_dsi_quality_flag", xarray=True)

        if qf_i is None:
            print(f"skip EFD 64 Hz quality load failed: {trange_i}")
            continue

        qf_i = qf_i.sortby("time").sel(time=slice(start, end))

        if qf_i.sizes.get("time", 0) == 0:
            print(f"skip empty EFD 64 Hz quality data: {trange_i}")
            continue

        qf_list.append(qf_i)

    if len(qf_list) == 0:
        raise ValueError("有効区間内で EFD 64 Hz quality flag を読み込めなかった。")

    E64_data_dsi_quality_flag = xr.concat(qf_list, dim="time").sortby("time")
    E64_data_dsi_quality_flag = _drop_duplicate_times(E64_data_dsi_quality_flag)
    E64_data_dsi_quality_flag.name = "E64_data_dsi_quality_flag"

    if save_path is not None:
        # 元CDF由来の attrs には dict など NetCDF に直列化できない値が含まれるため、
        # キャッシュ保存用には値と座標だけを持つ軽い DataArray に作り直す。
        qf_cache = xr.DataArray(
            E64_data_dsi_quality_flag.values,
            coords={"time": E64_data_dsi_quality_flag["time"].values},
            dims=("time",),
            name="E64_data_dsi_quality_flag",
            attrs={
                "long_name": "PWE-EFD 64 Hz DSI waveform quality flag",
                "analysis_ranges": "Only merged_ranges_df intervals were loaded.",
                "pad_seconds": float(pd.Timedelta(pad).total_seconds()),
            },
        )
        ds_qf = qf_cache.to_dataset(name="E64_data_dsi_quality_flag")
        ds_qf.to_netcdf(save_path)
        print(f"saved EFD 64 Hz quality flag: {save_path}")

    return E64_data_dsi_quality_flag


E64_quality_save_path = make_efd64_quality_save_path(
    merged_ranges_df,
    time_range,
)

force_reload_E64_quality = False

E64_data_dsi_quality_flag = load_efd64_quality_for_ranges(
    merged_ranges_df,
    pad=pd.Timedelta(seconds=2),
    no_update=False,
    use_cache=True,
    force_reload=force_reload_E64_quality,
    save_path=E64_quality_save_path,
)

# pad 部分を後段の bad 判定へ混ぜないため、merged_ranges_df 内だけに戻す。
mask_E64_in_merged_ranges = xr.DataArray(
    make_mask_in_ranges(E64_data_dsi_quality_flag["time"].values, merged_ranges_df),
    coords={"time": E64_data_dsi_quality_flag["time"]},
    dims=("time",),
)

E64_data_dsi_quality_flag = E64_data_dsi_quality_flag.where(
    mask_E64_in_merged_ranges,
    drop=True,
)

print(E64_data_dsi_quality_flag)
print(f"E64 quality samples in merged_ranges_df: {E64_data_dsi_quality_flag.sizes.get('time', 0)}")
print(f"E64 quality cache path: {E64_quality_save_path}")


# 第二段階検証

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr


def keep_true_runs_1d(
    cond,
    dim="time",
    min_run_length=2,
    check_time_gap=True,
    gap_factor=1.5,
):
    """
    1D boolean DataArray について、True が min_run_length 点以上
    連続する区間だけ True として残す。

    Parameters
    ----------
    cond : xr.DataArray
        1D boolean DataArray
    dim : str
        time dimension name
    min_run_length : int
        何点以上連続した True を残すか
    check_time_gap : bool
        True の場合、時刻 gap が通常 dt から大きく外れると別 run とみなす
    gap_factor : float
        median dt の gap_factor 倍より大きい gap で run を切る

    Returns
    -------
    keep_da : xr.DataArray
        cond と同じ time 軸を持つ boolean DataArray
    """

    cond = cond.fillna(False).astype(bool)
    values = cond.values.astype(bool)

    n = len(values)
    keep = np.zeros(n, dtype=bool)

    if n == 0:
        return xr.DataArray(
            keep,
            coords={dim: cond[dim]},
            dims=(dim,),
            name="consecutive_true_mask",
        )

    times = pd.DatetimeIndex(pd.to_datetime(cond[dim].values))

    if check_time_gap and n >= 2:
        t_ns = times.asi8
        dt_sec = np.diff(t_ns) * 1e-9
        dt_sec_valid = dt_sec[np.isfinite(dt_sec) & (dt_sec > 0)]

        if len(dt_sec_valid) > 0:
            dt_ref = np.median(dt_sec_valid)
            max_gap_sec = gap_factor * dt_ref
        else:
            max_gap_sec = np.inf
    else:
        max_gap_sec = np.inf

    i = 0

    while i < n:
        if not values[i]:
            i += 1
            continue

        j = i + 1

        while j < n and values[j]:
            if check_time_gap:
                gap_sec = (times[j] - times[j - 1]).total_seconds()
                if gap_sec > max_gap_sec:
                    break

            j += 1

        run_length = j - i

        if run_length >= min_run_length:
            keep[i:j] = True

        i = j

    keep_da = xr.DataArray(
        keep,
        coords={dim: cond[dim]},
        dims=(dim,),
        name="consecutive_true_mask",
    )

    return keep_da

In [ ]:
# MGF quality flag の bad は、前段で有効とした merged_ranges_df 内だけ評価する。
mask_qf_B_in_merged_ranges = xr.DataArray(
    make_mask_in_ranges(qf_B["time"].values, merged_ranges_df),
    coords={"time": qf_B["time"]},
    dims=("time",),
)

qf_B_for_bad_check = qf_B.where(mask_qf_B_in_merged_ranges, drop=True)

bad_mask_mgf_raw = qf_B_for_bad_check > 21

bad_mask_mgf = keep_true_runs_1d(
    bad_mask_mgf_raw,
    dim="time",
    min_run_length=4,     # 4点以上連続した bad のみ残す
    check_time_gap=True,
    gap_factor=1.5,
)

bad_times_mgf = qf_B_for_bad_check.time.where(bad_mask_mgf, drop=True)

print(f"MGF qf samples in merged_ranges_df: {qf_B_for_bad_check.sizes.get('time', 0)}")
print(f"raw MGF bad times: {int(bad_mask_mgf_raw.sum().values)}")
print(f"consecutive MGF bad times: {int(bad_mask_mgf.sum().values)}")
print(bad_times_mgf)


In [ ]:
bad_mask_efd_wave_raw = E64_data_dsi_quality_flag != 0

bad_mask_efd_wave = keep_true_runs_1d(
    bad_mask_efd_wave_raw,
    dim="time",
    min_run_length=10,
    check_time_gap=True,
    gap_factor=1.5,
)

bad_times_efd_wave = E64_data_dsi_quality_flag.time.where(bad_mask_efd_wave, drop=True)

print(f"EFD 64 Hz qf samples in merged_ranges_df: {E64_data_dsi_quality_flag.sizes.get('time', 0)}")
print(f"raw EFD wave bad times: {int(bad_mask_efd_wave_raw.sum().values)}")
print(f"consecutive EFD wave bad times: {int(bad_mask_efd_wave.sum().values)}")
print(bad_times_efd_wave)


In [ ]:
import numpy as np
import pandas as pd
import xarray as xr


def find_time_gap_ranges(
    time_da,
    threshold=pd.Timedelta(seconds=30),
    time_dim="time",
    reason="time_gap",
):
    """
    time 軸で threshold 以上の間隔が空いた箇所を bad gap として抽出する。

    Parameters
    ----------
    time_da : xr.DataArray
        time 座標。例: qf_B.time
    threshold : pd.Timedelta
        隣接時刻差がこの値以上なら gap とみなす。
    time_dim : str
        time dimension name.
    reason : str
        gap の reason 名。

    Returns
    -------
    gap_ranges_df : pd.DataFrame
        gap_start_time, gap_end_time, gap_duration_sec などを持つ表。
    """

    times = pd.DatetimeIndex(pd.to_datetime(time_da.values)).sort_values()

    columns = [
        "gap_start_time",
        "gap_end_time",
        "gap_duration_sec",
        "gap_duration_min",
        "reason",
    ]

    if len(times) < 2:
        return pd.DataFrame(columns=columns)

    dt = times[1:] - times[:-1]
    gap_mask = dt >= threshold

    rows = []

    for i in np.where(gap_mask)[0]:
        gap_duration_sec = dt[i].total_seconds()

        rows.append({
            "gap_start_time": times[i],
            "gap_end_time": times[i + 1],
            "gap_duration_sec": gap_duration_sec,
            "gap_duration_min": gap_duration_sec / 60,
            "reason": reason,
        })

    gap_ranges_df = pd.DataFrame(rows, columns=columns)

    if len(gap_ranges_df) == 0:
        return gap_ranges_df

    gap_ranges_df = gap_ranges_df.sort_values(
        ["gap_start_time", "gap_end_time"]
    ).reset_index(drop=True)

    return gap_ranges_df

In [ ]:
mgf_gap_ranges_df = find_time_gap_ranges(
    B64_data_dsi_qf.time,
    threshold=pd.Timedelta(milliseconds=63),
    reason="mgf_time_gap",
)

print(mgf_gap_ranges_df)

In [ ]:
efd_wave_gap_ranges_df = find_time_gap_ranges(
    E64_data_dsi_quality_flag.time,
    threshold=pd.Timedelta(seconds=30),
    reason="efd_wave_time_gap",
)

print(efd_wave_gap_ranges_df)


In [ ]:
import pandas as pd
import numpy as np


# ============================================================
# helper functions
# ============================================================

def _as_datetime_index_from_time_da(time_da):
    """
    xarray DataArray / numpy array / list-like を DatetimeIndex に変換する。
    """
    if time_da is None:
        return pd.DatetimeIndex([])

    if hasattr(time_da, "values"):
        vals = time_da.values
    else:
        vals = time_da

    out = pd.DatetimeIndex(pd.to_datetime(vals))
    out = out.dropna().sort_values().unique()
    out = pd.DatetimeIndex(out)

    return out


def point_times_to_intervals(
    times,
    gap_factor=1.5,
):
    """
    bad_times のような点列を、連続点ごとに interval 化する。

    例:
      t0, t0+1/64s, t0+2/64s, ... を
      [t0, tN] の1区間にまとめる。

    単発 bad time は [t, t] として残る。
    """

    times = _as_datetime_index_from_time_da(times)

    if len(times) == 0:
        return pd.DataFrame(columns=["start_time", "end_time"])

    if len(times) == 1:
        return pd.DataFrame({
            "start_time": times,
            "end_time": times,
        })

    t_ns = times.asi8
    dt_sec = np.diff(t_ns) * 1e-9

    dt_valid = dt_sec[np.isfinite(dt_sec) & (dt_sec > 0)]

    if len(dt_valid) == 0:
        return pd.DataFrame({
            "start_time": times,
            "end_time": times,
        })

    dt_ref = np.median(dt_valid)
    max_gap = gap_factor * dt_ref

    is_continuous = dt_sec <= max_gap

    break_points = np.where(~is_continuous)[0] + 1
    idx_groups = np.split(np.arange(len(times)), break_points)

    intervals = []

    for idx in idx_groups:
        intervals.append((times[idx[0]], times[idx[-1]]))

    return pd.DataFrame(intervals, columns=["start_time", "end_time"])


def gap_df_to_intervals(gap_df):
    """
    gap DataFrame を start_time, end_time の DataFrame に変換する。

    対応列:
      - missing_start_time, missing_end_time
      - gap_start_time, gap_end_time
    """

    if gap_df is None or len(gap_df) == 0:
        return pd.DataFrame(columns=["start_time", "end_time"])

    if {"missing_start_time", "missing_end_time"}.issubset(gap_df.columns):
        start_col = "missing_start_time"
        end_col = "missing_end_time"
    elif {"gap_start_time", "gap_end_time"}.issubset(gap_df.columns):
        start_col = "gap_start_time"
        end_col = "gap_end_time"
    else:
        raise ValueError(
            "gap_df must have either "
            "['missing_start_time', 'missing_end_time'] or "
            "['gap_start_time', 'gap_end_time'] columns."
        )

    out = pd.DataFrame({
        "start_time": pd.to_datetime(gap_df[start_col]),
        "end_time": pd.to_datetime(gap_df[end_col]),
    })

    out = out.dropna()
    out = out[out["end_time"] >= out["start_time"]]

    return out.reset_index(drop=True)


def merge_intervals(intervals_df):
    """
    start_time, end_time を持つ interval DataFrame を merge する。
    """

    if intervals_df is None or len(intervals_df) == 0:
        return pd.DataFrame(columns=["start_time", "end_time"])

    df = intervals_df.dropna().copy()
    df = df[df["end_time"] >= df["start_time"]]
    df = df.sort_values("start_time").reset_index(drop=True)

    if len(df) == 0:
        return pd.DataFrame(columns=["start_time", "end_time"])

    starts = pd.DatetimeIndex(pd.to_datetime(df["start_time"])).to_numpy()
    ends = pd.DatetimeIndex(pd.to_datetime(df["end_time"])).to_numpy()

    merged = []

    cur_start = starts[0]
    cur_end = ends[0]

    for s, e in zip(starts[1:], ends[1:]):
        if s <= cur_end:
            cur_end = max(cur_end, e)
        else:
            merged.append((cur_start, cur_end))
            cur_start = s
            cur_end = e

    merged.append((cur_start, cur_end))

    return pd.DataFrame({
        "start_time": pd.to_datetime([m[0] for m in merged]),
        "end_time": pd.to_datetime([m[1] for m in merged]),
    })


def make_forbidden_intervals_for_mono(
    bad_times,
    bad_times_mgf,
    bad_times_spin_angle,
    bad_times_efd_wave,
    LEPe_gap_ranges_df=None,
    LEPi_gap_ranges_df=None,
    mgf_gap_ranges_df=None,
    efd_wave_gap_ranges_df=None,
    att_gap_ranges_df=None,
    merge=True,
):
    """
    第二段階用の bad times / gap ranges をまとめて禁止 interval にする。
    """

    dfs = []

    bad_efd_intervals = point_times_to_intervals(bad_times)
    bad_mgf_intervals = point_times_to_intervals(bad_times_mgf)
    bad_spin_angle_intervals = point_times_to_intervals(bad_times_spin_angle)
    bad_efd_wave_intervals = point_times_to_intervals(bad_times_efd_wave)

    if len(bad_efd_intervals) > 0:
        dfs.append(bad_efd_intervals)

    if len(bad_mgf_intervals) > 0:
        dfs.append(bad_mgf_intervals)

    if len(bad_spin_angle_intervals) > 0:
        dfs.append(bad_spin_angle_intervals)

    if len(bad_efd_wave_intervals) > 0:
        dfs.append(bad_efd_wave_intervals)

    lepe_intervals = gap_df_to_intervals(LEPe_gap_ranges_df)
    lepi_intervals = gap_df_to_intervals(LEPi_gap_ranges_df)
    mgf_intervals = gap_df_to_intervals(mgf_gap_ranges_df)
    efd_intervals = gap_df_to_intervals(efd_wave_gap_ranges_df)
    att_intervals = gap_df_to_intervals(att_gap_ranges_df)

    if len(lepe_intervals) > 0:
        dfs.append(lepe_intervals)

    if len(lepi_intervals) > 0:
        dfs.append(lepi_intervals)

    if len(mgf_intervals) > 0:
        dfs.append(mgf_intervals)

    if len(efd_intervals) > 0:
        dfs.append(efd_intervals)

    if len(att_intervals) > 0:
        dfs.append(att_intervals)

    if len(dfs) == 0:
        return pd.DataFrame(columns=["start_time", "end_time"])

    forbidden_intervals_df = pd.concat(dfs, ignore_index=True)

    if merge:
        forbidden_intervals_df = merge_intervals(forbidden_intervals_df)

    return forbidden_intervals_df.reset_index(drop=True)


In [ ]:
def make_trimmed_time_ranges_by_forbidden(
    selected_times,
    forbidden_intervals_df,
    half_width=pd.Timedelta(minutes=15),
    eps=pd.Timedelta(nanoseconds=1),
):
    """
    selected_times ± half_width の時間範囲を作る。
    ただし forbidden intervals を含まないように、
    各 selected time の前後で最も近い forbidden interval まで切り詰める。
    """

    selected_times_pd = pd.DatetimeIndex(
        pd.to_datetime(selected_times)
    ).sort_values()

    if len(selected_times_pd) == 0:
        return []

    start0 = selected_times_pd - half_width
    end0   = selected_times_pd + half_width

    if forbidden_intervals_df is None or len(forbidden_intervals_df) == 0:
        return list(zip(start0, end0))

    forbidden_intervals_df = forbidden_intervals_df.sort_values("start_time").reset_index(drop=True)

    f_starts = pd.DatetimeIndex(pd.to_datetime(forbidden_intervals_df["start_time"]))
    f_ends   = pd.DatetimeIndex(pd.to_datetime(forbidden_intervals_df["end_time"]))

    t_np      = selected_times_pd.to_numpy()
    start_np  = start0.to_numpy()
    end_np    = end0.to_numpy()
    fs_np     = f_starts.to_numpy()
    fe_np     = f_ends.to_numpy()

    # 各 selected time に対して、start <= t となる最後の forbidden interval
    i_prev = np.searchsorted(fs_np, t_np, side="right") - 1
    valid_prev = i_prev >= 0

    inside_forbidden = np.zeros(len(t_np), dtype=bool)

    # selected time 自体が forbidden interval 内なら除外
    inside_forbidden[valid_prev] = (
        t_np[valid_prev] <= fe_np[i_prev[valid_prev]]
    )

    # 前側の forbidden interval で start を切る
    start_trim = start_np.copy()

    use_prev_trim = np.zeros(len(t_np), dtype=bool)
    use_prev_trim[valid_prev] = (
        fe_np[i_prev[valid_prev]] > start_np[valid_prev]
    )

    start_trim[use_prev_trim] = (
        fe_np[i_prev[use_prev_trim]] + np.timedelta64(eps.value, "ns")
    )

    # 後側の forbidden interval で end を切る
    i_next = i_prev + 1
    valid_next = i_next < len(fs_np)

    end_trim = end_np.copy()

    use_next_trim = np.zeros(len(t_np), dtype=bool)
    use_next_trim[valid_next] = (
        fs_np[i_next[valid_next]] < end_np[valid_next]
    )

    end_trim[use_next_trim] = (
        fs_np[i_next[use_next_trim]] - np.timedelta64(eps.value, "ns")
    )

    valid = (~inside_forbidden) & (end_trim > start_trim)

    start_out = pd.to_datetime(start_trim[valid])
    end_out   = pd.to_datetime(end_trim[valid])

    return list(zip(start_out, end_out))


def merge_time_ranges(
    time_ranges,
    max_duration=None,
):
    """
    overlapping time ranges を merge する。
    max_duration を指定した場合は、それを超える merge はしない。
    """

    if len(time_ranges) == 0:
        return []

    time_ranges = sorted(time_ranges)

    merged = []

    for start, end in time_ranges:
        if not merged:
            merged.append((start, end))
            continue

        prev_start, prev_end = merged[-1]

        is_overlapping = start <= prev_end

        candidate_start = prev_start
        candidate_end = max(prev_end, end)

        if max_duration is None:
            ok_duration = True
        else:
            ok_duration = (candidate_end - candidate_start) <= max_duration

        if is_overlapping and ok_duration:
            merged[-1] = (candidate_start, candidate_end)
        else:
            merged.append((start, end))

    return merged

In [ ]:
bad_spin_angle_intervals_df = point_times_to_intervals(bad_times_spin_angle)

print(f"spin-angle bad intervals: {len(bad_spin_angle_intervals_df)}")
bad_spin_angle_intervals_df


In [ ]:
import time

t0_clock = time.perf_counter()

# ============================================================
# forbidden intervals:
# EFD spec bad + MGF bad + spin angle bad + EFD wave bad
# + LEPe gap + LEPi gap + MGF gap + EFD wave gap + ATT gap
# ============================================================

forbidden_intervals_mono_df = make_forbidden_intervals_for_mono(
    bad_times=bad_times,
    bad_times_mgf=bad_times_mgf,
    bad_times_spin_angle=bad_times_spin_angle,
    bad_times_efd_wave=bad_times_efd_wave,
    LEPe_gap_ranges_df=LEPe_omniflux_gap_ranges_df,
    LEPi_gap_ranges_df=LEPi_omniflux_gap_ranges_df,
    mgf_gap_ranges_df=mgf_gap_ranges_df,
    efd_wave_gap_ranges_df=efd_wave_gap_ranges_df,
    att_gap_ranges_df=att_nan_intervals_df,
    merge=True,
)

elapsed = time.perf_counter() - t0_clock

print(f"forbidden intervals after merge: {len(forbidden_intervals_mono_df)}")
print(f"elapsed: {elapsed:.2f} sec")
forbidden_intervals_mono_df


In [ ]:
# 確認用
bad_times_efd_pd = _as_datetime_index_from_time_da(bad_times)
bad_times_mgf_pd = _as_datetime_index_from_time_da(bad_times_mgf)
bad_times_spin_angle_pd = _as_datetime_index_from_time_da(bad_times_spin_angle)
bad_times_efd_wave_pd = _as_datetime_index_from_time_da(bad_times_efd_wave)

print(f"EFD spec bad_times: {len(bad_times_efd_pd)}")
print(f"MGF qf bad_times: {len(bad_times_mgf_pd)}")
print(f"Spin angle bad_times: {len(bad_times_spin_angle_pd)}")
print(f"EFD wave qf bad_times: {len(bad_times_efd_wave_pd)}")
print(f"LEPe gaps: {len(LEPe_omniflux_gap_ranges_df) if LEPe_omniflux_gap_ranges_df is not None else 0}")
print(f"LEPi gaps: {len(LEPi_omniflux_gap_ranges_df) if LEPi_omniflux_gap_ranges_df is not None else 0}")
print(f"MGF time gaps: {len(mgf_gap_ranges_df) if mgf_gap_ranges_df is not None else 0}")
print(f"EFD wave time gaps: {len(efd_wave_gap_ranges_df) if efd_wave_gap_ranges_df is not None else 0}")
print(f"ATT gaps: {len(att_nan_intervals_df) if att_nan_intervals_df is not None else 0}")


# ============================================================
# selected_times_mono ±15 min を forbidden intervals で切る
# ============================================================

t0_clock = time.perf_counter()

time_ranges_mono = make_trimmed_time_ranges_by_forbidden(
    selected_times=selected_times_mono,
    forbidden_intervals_df=forbidden_intervals_mono_df,
    half_width=pd.Timedelta(minutes=15),
    eps=pd.Timedelta(nanoseconds=1),
)

print(f"time ranges before merge: {len(time_ranges_mono)}")


# ============================================================
# Merge overlapping time ranges
# ============================================================

merged_ranges_mono = merge_time_ranges(
    time_ranges_mono,
    max_duration=None,
)


# ============================================================
# DataFrame 化
# ============================================================

merged_ranges_mono_df = pd.DataFrame(
    merged_ranges_mono,
    columns=["start_time", "end_time"]
)

if len(merged_ranges_mono_df) > 0:
    merged_ranges_mono_df["duration_minutes"] = (
        merged_ranges_mono_df["end_time"] - merged_ranges_mono_df["start_time"]
    ).dt.total_seconds() / 60

    # 30分を超える時間範囲のみ有効にする
    merged_ranges_mono_df = merged_ranges_mono_df[
        merged_ranges_mono_df["duration_minutes"] > 30
    ].reset_index(drop=True)
else:
    merged_ranges_mono_df["duration_minutes"] = []

elapsed = time.perf_counter() - t0_clock

print(f"元の selected_times_mono 数: {len(selected_times_mono)}")
print(f"30分超のマージ後時間範囲数: {len(merged_ranges_mono_df)}")
print(f"elapsed: {elapsed:.2f} sec")
print("\n有効なマージ時間範囲:")
print(merged_ranges_mono_df)


In [ ]:
from pathlib import Path
from tqdm.auto import tqdm
import numpy as np
import pandas as pd
from matplotlib.colors import Normalize
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

path_base_save_plot = f'/mnt/j/statistical_analysis_arase/preanalysis'
path_base_save_plot = Path(path_base_save_plot + f'/PWE-EFD_spec_after_B')
os.makedirs(path_base_save_plot, exist_ok=True)

def edges_numeric(arr):
    arr = np.asarray(arr, dtype=float)
    if arr.size < 2:
        return np.concatenate((arr, arr + 1.0))
    d = np.diff(arr) / 2.0
    return np.concatenate(([arr[0] - d[0]], arr[:-1] + d, [arr[-1] + d[-1]]))

vmax = np.log10(1E1)
vmin = np.log10(1e-4)

for idx, row in tqdm(merged_ranges_mono_df.iterrows(), total=len(merged_ranges_mono_df), desc="Saving PWE-EFD spec plots"):
    start = row["start_time"]
    end = row["end_time"]

    if pd.isna(start) or pd.isna(end):
        continue

    subset_range = efd_spectra_qf.sel(time=slice(start, end))

    if subset_range.sizes.get("time", 0) < 2:
        continue

    times = pd.to_datetime(subset_range.time.values)

    if "spec_bins" in subset_range.coords:
        freqs = subset_range["spec_bins"].values
    elif "v" in subset_range.coords:
        freqs = subset_range["v"].values
    else:
        freqs = np.arange(subset_range.sizes["v_dim"])

    data_range = subset_range.values
    data_range = np.where(
        np.isfinite(data_range) & (data_range > 0),
        data_range,
        np.nan
    )

    logdata_range = np.log10(data_range)

    if np.isnan(logdata_range).all():
        continue

    t_num = mdates.date2num(times.to_pydatetime())
    t_edges = edges_numeric(t_num)
    f_edges = edges_numeric(freqs)

    fig_i, ax_i = plt.subplots(figsize=(12, 4.5))

    pcm_i = ax_i.pcolormesh(
        t_edges,
        f_edges,
        logdata_range.T,
        shading="auto",
        cmap="turbo",
        norm=Normalize(vmin=vmin, vmax=vmax),
    )

    f_cH_plot = f_cH.sel(time=slice(start, end))
    f_cHe_plot = f_cHe.sel(time=slice(start, end))
    f_cO_plot = f_cO.sel(time=slice(start, end))

    ax_i.plot(f_cH_plot.time, f_cH_plot.data, lw=1, c='white', linestyle='solid')
    ax_i.plot(f_cHe_plot.time, f_cHe_plot.data, lw=1, c='magenta', linestyle='solid')
    ax_i.plot(f_cO_plot.time, f_cO_plot.data, lw=1, c='yellow', linestyle='solid')

    ax_i.set_xlabel("Time")
    ax_i.xaxis_date()
    ax_i.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    ax_i.set_ylabel("Frequency [Hz]")
    ax_i.set_title(
        f"PWE-EFD spec ({start:%Y-%m-%d %H:%M:%S} - {end:%Y-%m-%d %H:%M:%S})"
    )
    ax_i.set_yscale("log")
    ax_i.set_ylim(3, 32)
    ax_i.set_xlim(start.to_pydatetime(), end.to_pydatetime())

    fig_i.colorbar(pcm_i, ax=ax_i, label=r"log10 PSD [$\mathrm{(mV/m)}^2/\mathrm{Hz}$]")
    plt.tight_layout()

    # =========================
    # save figure
    # =========================

    # start の月・日で階層フォルダを作る
    save_dir = path_base_save_plot / f"{start:%Y}" / f"{start:%m}"
    save_dir.mkdir(parents=True, exist_ok=True)

    filename = (
        f"efd_spectra_qf_"
        f"{start:%Y%m%d_%H%M%S}_to_{end:%Y%m%d_%H%M%S}.png"
    )

    save_path = save_dir / filename

    fig_i.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.close(fig_i)

    print(f"saved: {save_path}")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.colors import Normalize
from pathlib import Path
from tqdm.auto import tqdm


def edges_numeric(arr):
    arr = np.asarray(arr, dtype=float)
    if arr.size < 2:
        return np.concatenate((arr, arr + 1.0))
    d = np.diff(arr) / 2.0
    return np.concatenate(([arr[0] - d[0]], arr[:-1] + d, [arr[-1] + d[-1]]))


# ============================================================
# save base path
# ============================================================

path_base_save_plot = f'/mnt/j/statistical_analysis_arase/preanalysis'
path_base_save_plot = Path(path_base_save_plot + f'/MGF_spec_after_B')
os.makedirs(path_base_save_plot, exist_ok=True)



# ============================================================
# plot settings
# ============================================================

vmin = -4
vmax = -1

for idx, row in tqdm(merged_ranges_mono_df.iterrows(), total=len(merged_ranges_mono_df), desc="Saving MGF spec plots"):

    start = pd.to_datetime(row["start_time"])
    end   = pd.to_datetime(row["end_time"])

    if pd.isna(start) or pd.isna(end):
        continue

    f_cH_plot = f_cH.sel(time=slice(start, end))
    f_cHe_plot = f_cHe.sel(time=slice(start, end))
    f_cO_plot = f_cO.sel(time=slice(start, end))

    # xarray sel 用
    B64_spectra_qf_analysis = B64_spectra_qf.sel(
        time=slice(start, end)
    )

    # time が少なすぎる場合は skip
    if B64_spectra_qf_analysis.sizes.get("time", 0) < 2:
        print(f"skip: too few time points: {start} - {end}")
        continue

    data = B64_spectra_qf_analysis.values
    data = np.where(np.isfinite(data) & (data > 0), data, np.nan)

    if np.isnan(data).all():
        print(f"skip: all NaN: {start} - {end}")
        continue

    logdata = np.log10(data)

    times = pd.DatetimeIndex(pd.to_datetime(B64_spectra_qf_analysis.time.values))
    t_num = mdates.date2num(times.to_pydatetime())
    t_edges = edges_numeric(t_num)

    freqs = B64_spectra_qf_analysis.spec_bins.values
    f_edges = edges_numeric(freqs)

    fig, ax = plt.subplots(figsize=(12, 4.5))

    pcm = ax.pcolormesh(
        t_edges,
        f_edges,
        logdata.T,
        shading="auto",
        cmap="turbo",
        norm=Normalize(vmin=vmin, vmax=vmax),
    )

    ax.plot(f_cH_plot.time, f_cH_plot.data, lw=1, c='white', linestyle='solid')
    ax.plot(f_cHe_plot.time, f_cHe_plot.data, lw=1, c='magenta', linestyle='solid')
    ax.plot(f_cO_plot.time, f_cO_plot.data, lw=1, c='yellow', linestyle='solid')

    ax.xaxis_date()
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))

    ax.set_yscale("log")
    ax.set_ylim(3, 32)

    ax.set_xlim(start.to_pydatetime(), end.to_pydatetime())

    ax.set_xlabel("Time")
    ax.set_ylabel("Frequency [Hz]")
    ax.set_title(
        f"MGF 64hz total spec "
        f"({start:%Y-%m-%d %H:%M:%S} - {end:%Y-%m-%d %H:%M:%S})"
    )

    fig.colorbar(
        pcm,
        ax=ax,
        label=r"log10 PSD [$\mathrm{nT}^2/\mathrm{Hz}$]"
    )

    plt.tight_layout()

    # ========================================================
    # save
    # ========================================================

    save_dir = path_base_save_plot / f"{start:%Y}" / f"{start:%m}"
    save_dir.mkdir(parents=True, exist_ok=True)

    filename = (
        f"MGF_B64_spectra_"
        f"{start:%Y%m%d_%H%M%S}_to_{end:%Y%m%d_%H%M%S}.png"
    )

    save_path = save_dir / filename

    fig.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.close(fig)

    print(f"saved: {save_path}")


# 最終有効時間の保存

第二段階検証後の `merged_ranges_mono_df` を、人が確認しやすい CSV と、再利用しやすい JSON、条件確認用の summary txt として保存する。


In [ ]:
from pathlib import Path
import json
import pandas as pd
import numpy as np

# ============================================================
# save final valid time ranges
# ============================================================

save_dir_valid_times = Path("/mnt/j/statistical_analysis_arase/preanalysis/valid_time_ranges")
save_dir_valid_times.mkdir(parents=True, exist_ok=True)

if "merged_ranges_mono_df" not in globals():
    raise NameError("merged_ranges_mono_df が未定義です。第二段階検証を先に実行してください。")

valid_time_ranges_df = merged_ranges_mono_df.copy()

if len(valid_time_ranges_df) > 0:
    valid_time_ranges_df["start_time"] = pd.to_datetime(valid_time_ranges_df["start_time"])
    valid_time_ranges_df["end_time"] = pd.to_datetime(valid_time_ranges_df["end_time"])
    valid_time_ranges_df = valid_time_ranges_df.sort_values("start_time").reset_index(drop=True)
    valid_time_ranges_df.insert(0, "range_id", np.arange(1, len(valid_time_ranges_df) + 1))

    valid_time_ranges_df["duration_sec"] = (
        valid_time_ranges_df["end_time"] - valid_time_ranges_df["start_time"]
    ).dt.total_seconds()
    valid_time_ranges_df["duration_min"] = valid_time_ranges_df["duration_sec"] / 60

    t0_save = valid_time_ranges_df["start_time"].min()
    t1_save = valid_time_ranges_df["end_time"].max()
else:
    t0_save = pd.to_datetime(time_range[0]) if "time_range" in globals() else pd.Timestamp("1970-01-01")
    t1_save = pd.to_datetime(time_range[-1]) if "time_range" in globals() else pd.Timestamp("1970-01-01")

base_name = (
    f"Arase_valid_time_ranges_"
    f"{t0_save:%Y%m%d_%H%M%S}_to_{t1_save:%Y%m%d_%H%M%S}"
)

csv_path = save_dir_valid_times / f"{base_name}.csv"
json_path = save_dir_valid_times / f"{base_name}.json"
summary_path = save_dir_valid_times / f"{base_name}_summary.txt"

# CSV: 一番見やすい確認用
valid_time_ranges_df.to_csv(csv_path, index=False)

# JSON: 後で Python から読み直しやすい形式
json_records = []
for _, row in valid_time_ranges_df.iterrows():
    json_records.append({
        "range_id": int(row["range_id"]),
        "start_time": pd.to_datetime(row["start_time"]).isoformat(),
        "end_time": pd.to_datetime(row["end_time"]).isoformat(),
        "duration_sec": float(row["duration_sec"]),
        "duration_min": float(row["duration_min"]),
    })

metadata = {
    "description": "Final valid time ranges after Arase event preanalysis.",
    "source_notebook": "Arase_event_preanalysis.ipynb",
    "time_range_input": time_range if "time_range" in globals() else None,
    "n_valid_ranges": int(len(valid_time_ranges_df)),
    "total_duration_min": float(valid_time_ranges_df["duration_min"].sum()) if len(valid_time_ranges_df) > 0 else 0.0,
    "selection_summary": {
        "n_selected_times": int(len(selected_times)) if "selected_times" in globals() else None,
        "n_selected_times_mono": int(len(selected_times_mono)) if "selected_times_mono" in globals() else None,
        "n_forbidden_intervals_mono": int(len(forbidden_intervals_mono_df)) if "forbidden_intervals_mono_df" in globals() else None,
    },
    "conditions": [
        "PWE-EFD spec high-power condition in 3--32 Hz",
        "LEPe/LEPi omniflux bad intervals excluded",
        "ATT NaN intervals excluded",
        "MGF PSD monotonic-decreasing condition applied",
        "MGF quality bad intervals excluded",
        "EFD waveform quality bad intervals excluded",
        "B spin-plane angle condition applied",
        "MGF/EFD waveform time gaps excluded",
        "Final ranges longer than 30 minutes retained",
    ],
    "ranges": json_records,
}

with open(json_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

summary_lines = [
    "Arase event preanalysis: final valid time ranges",
    "=" * 56,
    f"source_notebook: Arase_event_preanalysis.ipynb",
    f"input time_range: {metadata['time_range_input']}",
    f"n_valid_ranges: {metadata['n_valid_ranges']}",
    f"total_duration_min: {metadata['total_duration_min']:.3f}",
    f"n_selected_times: {metadata['selection_summary']['n_selected_times']}",
    f"n_selected_times_mono: {metadata['selection_summary']['n_selected_times_mono']}",
    f"n_forbidden_intervals_mono: {metadata['selection_summary']['n_forbidden_intervals_mono']}",
    "",
    "conditions:",
]

summary_lines.extend([f"- {cond}" for cond in metadata["conditions"]])
summary_lines.append("")
summary_lines.append("valid ranges:")

if len(valid_time_ranges_df) == 0:
    summary_lines.append("- none")
else:
    for _, row in valid_time_ranges_df.iterrows():
        summary_lines.append(
            f"- {int(row['range_id']):03d}: "
            f"{row['start_time']:%Y-%m-%d %H:%M:%S} -- "
            f"{row['end_time']:%Y-%m-%d %H:%M:%S} "
            f"({row['duration_min']:.2f} min)"
        )

summary_path.write_text("\n".join(summary_lines) + "\n", encoding="utf-8")

print(f"saved CSV    : {csv_path}")
print(f"saved JSON   : {json_path}")
print(f"saved summary: {summary_path}")
print(f"n_valid_ranges = {len(valid_time_ranges_df)}")
print(f"total_duration_min = {metadata['total_duration_min']:.3f}")

valid_time_ranges_df
